# ARMOR-FLD — Face Landmark Detection Under Adverse Conditions
### *Adverse-condition Reliability-Modulated ORFormer* with Prior-Guided Test-Time Adaptation
**Sayan Rakshit** · single-notebook research code for Google Colab (T4 / L4 / A100)

This notebook turns the SCRFD + ORFormer + STAR-loss proposal into a **novel, publishable method** and trains / evaluates it end-to-end on WFLW.

| | Original proposal (SCRFD + ORFormer + STAR) | **ARMOR-FLD (this notebook)** |
|---|---|---|
| Training corruptions | random occlusion only | **Degradation Bank**: occlusion, blur, low-light+noise, haze/rain, compression — each with a *per-pixel unreliability map* |
| ORFormer's patch score α | learned implicitly (occlusion only) | **reliability-supervised** (BCE vs. unreliability map) + **degradation-type head** that conditions the messenger-token recovery + reliability-biased attention |
| Prior → landmark net | plain 1×1-conv concat | **reliability-gated fusion**: the network trusts the clean-face prior exactly where the image is unreliable (learned gate + κ·(1−r)) |
| Loss | STAR | **STAR-V**: visibility-weighted STAR + a **per-landmark visibility head** (needed for the failure-mode handling in Ch. 5) |
| Test time | none | **PG-TTA**: label-free, source-free adaptation of normalisation layers using (i) consistency with the frozen clean-face prior, (ii) flip equivariance, (iii) heatmap entropy — with the number of steps gated by a **VQ shift detector** (quantisation error of the clean-face codebook) |
| Benchmark | WFLW / COFW / 300W | + **WFLW-C** protocol: 5 degradation families × 5 severities, NME / FR / AUC per cell, plus visibility accuracy |

Pipeline: `SCRFD (ONNX, 640²) → 1.3× square crop (128²) → [frozen clean-face VQ prior ⟶ R-ORFormer ⟶ H_rec, α] + [stacked hourglass with reliability-gated fusion] → STAR-V soft-argmax landmarks + visibility → exact affine back-projection onto the original image`.

**How to use**
1. Runtime → Change runtime type → **GPU**.
2. Set `MODE = "QUICK"` (≈20 min on a T4, 1,500-face subset of WFLW, proves everything runs end to end) or `MODE = "FULL"` (all 7,500 faces; ~1.5–2.5 h on a T4, ~45 min on an A100).
3. Run all cells. Checkpoints and result tables are written to Google Drive (`cfg["out_dir"]`) so a disconnect does not lose work.

If the WFLW download fails (Google-Drive quota is common), download `WFLW_images.tar.gz` yourself from the WFLW page, put it in `MyDrive/WFLW/` and re-run the data cell. The notebook otherwise falls back to **synthetic faces** (it prints `synthetic mode: True`) — that only checks the code; numbers from a synthetic run (e.g. NME < 1 %) mean nothing for WFLW.

## 0 · Environment

In [ ]:
import os, sys, subprocess, time, json, math, glob
IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)
try:
    import torch
    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print(e)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxscript", "onnxruntime-gpu", "gdown", "tqdm"], check=False)
import onnxruntime as ort; print("onnxruntime", ort.__version__, ort.get_available_providers())

In [ ]:
# Google Drive for checkpoints / results (optional but recommended)
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/ARMOR_FLD"
else:
    OUT_DIR = "/content/ARMOR_FLD" if IN_COLAB else os.path.abspath("./ARMOR_FLD")
os.makedirs(OUT_DIR, exist_ok=True)
WORK = "/content" if IN_COLAB else os.path.abspath(".")
os.makedirs(f"{WORK}/armor", exist_ok=True)
os.chdir(WORK)
print("work dir:", WORK, "| outputs:", OUT_DIR)

## 1 · The `armor` package (all code is written to disk so you can also run it as plain scripts)

In [ ]:
%%writefile armor/__init__.py
"""ARMOR-FLD: Adverse-condition Reliability-Modulated ORFormer for facial landmark detection.

Novel pipeline built on top of SCRFD (detection) + ORFormer (clean-face prior) + STAR loss:

  1. Degradation Bank with *reliability supervision*  (armor.degrade)
  2. R-ORFormer: reliability- and degradation-type-supervised messenger recovery (armor.orformer)
  3. Reliability-gated heatmap fusion + per-landmark visibility head + STAR-V loss (armor.fld, armor.losses)
  4. Shift detector (mean predicted unreliability) gating Prior-Guided Test-Time Adaptation (armor.tta)
  5. WFLW-C adverse-condition benchmark protocol (armor.train)
"""
__version__ = "0.1.0"

In [ ]:
%%writefile armor/config.py
"""Experiment configuration presets.

QUICK  : sanity run that finishes on a free Colab T4 in well under an hour (subset of WFLW, few epochs).
FULL   : the real schedule (all 7,500 training faces) - several hours on a T4, ~1.5-2 h on an A100.

Everything is a plain dict so it can be edited inline in the notebook.
"""
import copy

N_LANDMARKS = 98          # WFLW
N_EDGES = 13              # edge/boundary heatmaps rendered from the 98 points (see armor.common.EDGE_GROUPS)
N_DEG_TYPES = 6           # clean, occlusion, blur, lowlight, weather, compression

BASE = dict(
    # ---------------- data ----------------
    data_root="/content/data/WFLW",   # where WFLW_images/ and WFLW_annotations/ live
    cache_dir="/content/cache",       # pre-cropped face cache (fast training on Colab)
    cache_size=160,                   # cached crop resolution (contains the 128 training crop + margin)
    cache_scale=1.6,                  # cached square side = cache_scale * max(box_w, box_h)
    crop_scale=1.3,                   # training / inference crop side = crop_scale * max(box_w, box_h)
    crop_size=128,                    # network input resolution (ORFormer branch sees crop_size // 2 = 64)
    heatmap_size=64,                  # landmark / edge heatmap resolution
    lmk_sigma=2.0,                    # gaussian sigma (heatmap pixels) for landmark heatmaps
    edge_sigma=1.0,                   # gaussian sigma for edge heatmaps
    train_subset=None,                # int -> use only the first N training faces (QUICK mode)
    test_subset=None,
    num_workers=2,
    # ---------------- augmentation ----------------
    aug_rot=30.0, aug_scale=0.10, aug_shift=0.05, aug_flip=0.5,
    aug_color=0.3,                    # brightness/contrast jitter strength
    p_degrade=0.85,                   # probability that the degradation bank is applied to a training crop
    # ---------------- VQ prior (stage A) ----------------
    code_dim=128, n_codes=512, enc_ch=(64, 128), vq_beta=0.25, vq_decay=0.99,
    a_epochs=12, a_lr=7e-4, a_bs=128,
    # ---------------- R-ORFormer (stage B) ----------------
    d_model=192, n_layers=3, n_heads=6, ffn_mult=4, drop=0.1,
    b_epochs=10, b_lr=2e-4, b_bs=64,
    w_code=1.0, w_hm=50.0, w_rel=1.0, w_type=0.5,
    # ---------------- Landmark net (stage C) ----------------
    hg_stacks=2, hg_depth=3, hg_ch=128,
    c_epochs=30, c_lr=1e-3, c_bs=32, c_wd=1e-5,
    finetune_prior=False,             # keep the clean-face prior frozen in stage C (ORFormer-style)
    softargmax_beta=10.0, softargmax_window=9,   # windowed soft-argmax (no centre bias, local covariance for STAR)
    w_star=1.0, w_awing=1.0, w_vis=0.5, w_edge=1.0,  # AWing forms the peaks fast; STAR-V refines them (sub-pixel + anisotropy)
    star_detach=True, star_vr=0.0, vis_floor=0.3,
    star_warmup=0.1,                  # fraction of stage-C steps over which the STAR-V weight ramps 0 -> w_star
    star_gate_px=4,                   # STAR-V only refines peaks already within 4 heat-map px of the target (0 = off)
    # ---------------- test-time adaptation ----------------
    tta_lr=1e-3, tta_max_steps=4, tta_online=False,
    tta_w_prior=1.0, tta_w_flip=1.0, tta_w_ent=0.1,
    tta_shift_mode="alpha",           # "alpha": R-ORFormer mean unreliability (default) | "vq": codebook quantisation error
    tta_z_per_step=0.5,               # one extra TTA step per +0.5 sigma of the shift score (clean faces stay at 0)
    tta_force_steps=None,             # int -> ignore the shift detector and always take this many steps (ablation)
    # ---------------- misc ----------------
    seed=0, amp=True, log_every=50,
    out_dir="/content/drive/MyDrive/ARMOR_FLD",  # checkpoints + results (Drive so they survive a disconnect)
)

QUICK = copy.deepcopy(BASE)
QUICK.update(dict(train_subset=1500, test_subset=600,
                  a_epochs=4, b_epochs=4, c_epochs=15,
                  n_codes=256, c_bs=32, log_every=20))

FULL = copy.deepcopy(BASE)
FULL.update(dict(a_epochs=30, b_epochs=20, c_epochs=60, n_codes=1024, code_dim=192))


def get_config(name="QUICK", **overrides):
    cfg = copy.deepcopy({"QUICK": QUICK, "FULL": FULL, "BASE": BASE}[name.upper()])
    cfg.update(overrides)
    return cfg

In [ ]:
%%writefile armor/common.py
"""Shared constants and geometry / heatmap helpers (numpy + OpenCV, no torch)."""
import math
import numpy as np
import cv2

# ----------------------------------------------------------------------------------------------
# WFLW 98-point conventions
# ----------------------------------------------------------------------------------------------
LEFT_EYE_OUTER, RIGHT_EYE_OUTER = 60, 72      # inter-ocular normalisation (WFLW protocol)

# horizontal-flip permutation (index i -> its mirrored landmark)
WFLW_FLIP = (
    list(range(32, -1, -1))                                   # 0-32  face contour
    + [46, 45, 44, 43, 42, 50, 49, 48, 47]                    # 33-41 left brow  -> right brow
    + [37, 36, 35, 34, 33, 41, 40, 39, 38]                    # 42-50 right brow -> left brow
    + [51, 52, 53, 54]                                        # 51-54 nose bridge
    + [59, 58, 57, 56, 55]                                    # 55-59 nose bottom
    + [72, 71, 70, 69, 68, 75, 74, 73]                        # 60-67 left eye -> right eye
    + [64, 63, 62, 61, 60, 67, 66, 65]                        # 68-75 right eye -> left eye
    + [82, 81, 80, 79, 78, 77, 76]                            # 76-82 outer upper lip
    + [87, 86, 85, 84, 83]                                    # 83-87 outer lower lip
    + [92, 91, 90, 89, 88]                                    # 88-92 inner upper lip
    + [95, 94, 93]                                            # 93-95 inner lower lip
    + [97, 96]                                                # 96-97 pupils
)
assert len(WFLW_FLIP) == 98 and sorted(WFLW_FLIP) == list(range(98))

# edge (boundary) groups rendered as poly-lines; (indices, closed?)
EDGE_GROUPS = [
    (list(range(0, 33)), False),            # 0  face contour
    (list(range(33, 42)), True),            # 1  left eyebrow (loop)
    (list(range(42, 51)), True),            # 2  right eyebrow (loop)
    ([51, 52, 53, 54], False),              # 3  nose bridge
    ([55, 56, 57, 58, 59], False),          # 4  nose bottom
    (list(range(60, 68)), True),            # 5  left eye
    (list(range(68, 76)), True),            # 6  right eye
    ([76, 77, 78, 79, 80, 81, 82], False),  # 7  outer upper lip
    ([82, 83, 84, 85, 86, 87, 76], False),  # 8  outer lower lip
    ([88, 89, 90, 91, 92], False),          # 9  inner upper lip
    ([92, 93, 94, 95, 88], False),          # 10 inner lower lip
    ([96], False),                          # 11 left pupil
    ([97], False),                          # 12 right pupil
]
N_EDGES = len(EDGE_GROUPS)

ATTR_NAMES = ["pose", "expression", "illumination", "makeup", "occlusion", "blur"]

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


# ----------------------------------------------------------------------------------------------
# Geometry: one float32 affine with an exact inverse for every crop / back-projection
# ----------------------------------------------------------------------------------------------
def crop_affine(center, side, out_size, rot_deg=0.0, flip=False):
    """2x3 matrix mapping *source image* coords -> *output crop* coords.

    The crop is the square of edge `side` (source pixels) centred on `center`, rotated by
    `rot_deg` (degrees, counter-clockwise in image coordinates) and resized to `out_size`.
    """
    cx, cy = float(center[0]), float(center[1])
    s = out_size / float(side)
    a = math.radians(rot_deg)
    cos, sin = math.cos(a) * s, math.sin(a) * s
    # rotate+scale about the centre, then translate the centre to the crop centre
    M = np.array([[cos, -sin, 0.0], [sin, cos, 0.0]], dtype=np.float64)
    t = np.array([cx, cy])
    M[:, 2] = (out_size - 1) / 2.0 - M[:, :2] @ t
    if flip:  # x' = (out_size-1) - x
        F = np.array([[-1.0, 0.0, out_size - 1.0], [0.0, 1.0, 0.0]])
        M = np.vstack([F @ np.vstack([M, [0, 0, 1]]), [0, 0, 1]])[:2]
    return M.astype(np.float32)


def invert_affine(M):
    return cv2.invertAffineTransform(M.astype(np.float32)).astype(np.float32)


def transform_points(pts, M):
    """Apply a 2x3 affine to an (N,2) array of points."""
    pts = np.asarray(pts, dtype=np.float32)
    return pts @ M[:, :2].T + M[:, 2]


def warp(img, M, out_size, border=cv2.BORDER_CONSTANT, value=(0, 0, 0)):
    return cv2.warpAffine(img, M, (out_size, out_size), flags=cv2.INTER_LINEAR,
                          borderMode=border, borderValue=value)


def square_from_box(box, scale):
    """(x1,y1,x2,y2) -> (center, side) of the square crop used by the pipeline."""
    x1, y1, x2, y2 = [float(v) for v in box[:4]]
    side = max(x2 - x1, y2 - y1) * scale
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0), side


# ----------------------------------------------------------------------------------------------
# Heatmap rendering
# ----------------------------------------------------------------------------------------------
def render_landmark_heatmaps(pts, size, sigma):
    """Gaussian heatmaps (N,size,size) float32 for points given in heatmap-pixel coordinates."""
    pts = np.asarray(pts, dtype=np.float32)
    xs = np.arange(size, dtype=np.float32)
    gx = np.exp(-((xs[None, :] - pts[:, 0:1]) ** 2) / (2 * sigma ** 2))  # (N,size)
    gy = np.exp(-((xs[None, :] - pts[:, 1:2]) ** 2) / (2 * sigma ** 2))
    hm = gy[:, :, None] * gx[:, None, :]
    return hm.astype(np.float32)


def render_edge_heatmaps(pts, size, sigma, groups=EDGE_GROUPS):
    """Boundary heatmaps (N_EDGES,size,size): exp(-d^2/2s^2) of the distance to the poly-line."""
    pts = np.asarray(pts, dtype=np.float32)
    shift = 4
    out = np.zeros((len(groups), size, size), dtype=np.float32)
    for g, (idx, closed) in enumerate(groups):
        canvas = np.zeros((size, size), dtype=np.uint8)
        p = pts[idx]
        if not np.all(np.isfinite(p)):
            continue
        q = np.round(p * (1 << shift)).astype(np.int32)
        if len(idx) == 1:
            cv2.circle(canvas, (int(q[0, 0]), int(q[0, 1])), 0, 255, -1, lineType=cv2.LINE_8, shift=shift)
        else:
            cv2.polylines(canvas, [q.reshape(-1, 1, 2)], closed, 255, 1, lineType=cv2.LINE_8, shift=shift)
        if canvas.max() == 0:
            continue
        dist = cv2.distanceTransform(255 - canvas, cv2.DIST_L2, 3)
        out[g] = np.exp(-(dist ** 2) / (2 * sigma ** 2))
    return out


def normalize_image(img_rgb_uint8):
    """HWC uint8 RGB -> CHW float32 normalised."""
    x = img_rgb_uint8.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return np.ascontiguousarray(x.transpose(2, 0, 1))


def denormalize_image(chw):
    x = np.asarray(chw).transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(x * 255.0, 0, 255).astype(np.uint8)

In [ ]:
%%writefile armor/degrade.py
"""Degradation Bank with *reliability supervision*.

Every degradation returns, besides the corrupted image, a per-pixel **unreliability map** u in [0,1]
(how little a local patch can be trusted) and the degradation **type**.  The maps are pooled onto the
ORFormer token grid (16x16) and become supervision for the per-patch reliability score alpha and for the
degradation-type head - this is what turns ORFormer's implicit "occlusion" notion into an explicit,
multi-type *reliability* notion (contribution 1 of ARMOR-FLD).

The same bank, driven deterministically by (type, severity, seed), defines the WFLW-C benchmark.
"""
import numpy as np
import cv2

DEG_TYPES = ["clean", "occlusion", "blur", "lowlight", "weather", "compression"]
TYPE_ID = {n: i for i, n in enumerate(DEG_TYPES)}


def _soft_mask(mask_uint8, k=7):
    m = cv2.GaussianBlur(mask_uint8.astype(np.float32), (k | 1, k | 1), 0)
    return np.clip(m, 0, 1)


def _smooth_field(rng, size, scale=4):
    """Low-frequency random field in [0,1]."""
    small = rng.random((scale, scale)).astype(np.float32)
    f = cv2.resize(small, (size, size), interpolation=cv2.INTER_CUBIC)
    f = (f - f.min()) / (f.max() - f.min() + 1e-6)
    return f


class DegradationBank:
    """All methods take/return HxWx3 uint8 RGB images and return (img, unreliability_map)."""

    def __init__(self, grid=16):
        self.grid = grid

    # ------------------------------------------------------------------ occlusion
    def occlusion(self, img, sev, rng):
        H, W = img.shape[:2]
        out = img.copy()
        mask = np.zeros((H, W), np.uint8)
        n = 1 if sev <= 2 else rng.integers(1, 3)
        for _ in range(n):
            frac = 0.12 + 0.07 * sev + rng.uniform(-0.03, 0.03)          # occluder size relative to crop
            w, h = int(W * frac * rng.uniform(0.7, 1.4)), int(H * frac * rng.uniform(0.7, 1.4))
            w, h = max(8, min(w, W - 2)), max(8, min(h, H - 2))
            cx, cy = rng.integers(w // 2, W - w // 2), rng.integers(h // 2, H - h // 2)
            m = np.zeros((H, W), np.uint8)
            shape = rng.integers(0, 3)
            if shape == 0:    # rectangle
                cv2.rectangle(m, (cx - w // 2, cy - h // 2), (cx + w // 2, cy + h // 2), 1, -1)
            elif shape == 1:  # ellipse
                cv2.ellipse(m, (int(cx), int(cy)), (w // 2, h // 2), float(rng.uniform(0, 180)), 0, 360, 1, -1)
            else:             # hand-like blob = union of ellipses
                for _ in range(4):
                    cv2.ellipse(m, (int(cx + rng.integers(-w // 3, w // 3 + 1)), int(cy + rng.integers(-h // 3, h // 3 + 1))),
                                (max(3, w // 3), max(3, h // 2)), float(rng.uniform(0, 180)), 0, 360, 1, -1)
            fill = rng.integers(0, 3)
            if fill == 0:     # solid random colour
                tex = np.empty_like(img); tex[:] = rng.integers(0, 256, 3)
            elif fill == 1:   # noise texture
                tex = rng.integers(0, 256, img.shape).astype(np.uint8)
            else:             # copy-paste of a (flipped) patch of the same image = realistic self-occluder
                tex = np.ascontiguousarray(img[::-1, ::-1])
                tex = np.roll(tex, (int(rng.integers(0, H)), int(rng.integers(0, W))), axis=(0, 1))
            out[m > 0] = tex[m > 0]
            mask |= m
        u = _soft_mask(mask, 5)
        return out, u

    # ------------------------------------------------------------------ blur
    def blur(self, img, sev, rng):
        H, W = img.shape[:2]
        k = int(3 + 4 * sev)                                       # 7 .. 23 px kernels at 128 px
        if rng.random() < 0.6:                                     # motion blur
            kern = np.zeros((k, k), np.float32); kern[k // 2, :] = 1.0
            Mrot = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1.0)
            kern = cv2.warpAffine(kern, Mrot, (k, k)); kern /= kern.sum() + 1e-8
            blurred = cv2.filter2D(img, -1, kern)
        else:                                                      # defocus
            sigma = 0.6 * sev + 0.4
            blurred = cv2.GaussianBlur(img, (0, 0), sigma)
        strength = min(1.0, (k - 3) / 20.0)
        if rng.random() < 0.5:                                     # regional blur (soft mask)
            field = _smooth_field(rng, H, 3)
            region = (field > np.quantile(field, 0.45)).astype(np.uint8)
            m = _soft_mask(region, 15)[..., None]
            out = (blurred * m + img * (1 - m)).astype(np.uint8)
            u = strength * m[..., 0]
        else:
            out, u = blurred, np.full((H, W), strength, np.float32)
        return out, u

    # ------------------------------------------------------------------ low light + sensor noise
    def lowlight(self, img, sev, rng):
        H, W = img.shape[:2]
        x = img.astype(np.float32) / 255.0
        gamma = 1.0 + 0.6 * sev
        gain = 1.0 - 0.12 * sev
        vign = 1.0
        if rng.random() < 0.5:                                     # vignette: darker at the borders
            yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
            r = np.sqrt(((xx - W / 2) / (W / 2)) ** 2 + ((yy - H / 2) / (H / 2)) ** 2)
            vign = np.clip(1.2 - 0.6 * r, 0.3, 1.0)[..., None]
        x = (x ** gamma) * gain * vign
        noise_sigma = (4 + 6 * sev) / 255.0
        x = x + rng.normal(0, noise_sigma, x.shape).astype(np.float32)            # read noise
        x = x + rng.normal(0, 1, x.shape).astype(np.float32) * noise_sigma * np.sqrt(np.clip(x, 0, 1))  # shot noise
        out = np.clip(x * 255.0, 0, 255).astype(np.uint8)
        base = min(0.9, 0.15 * sev)
        u = np.full((H, W), base, np.float32)
        if np.ndim(vign):
            u = u * (1.4 - vign[..., 0])                                            # darker corners = less reliable
        return out, np.clip(u, 0, 1).astype(np.float32)

    # ------------------------------------------------------------------ weather: haze + rain
    def weather(self, img, sev, rng):
        H, W = img.shape[:2]
        x = img.astype(np.float32)
        field = _smooth_field(rng, H, 4)
        t = 1.0 - (0.10 + 0.14 * sev) * (0.5 + 0.5 * field)          # transmission
        A = rng.uniform(200, 255)
        out = x * t[..., None] + A * (1 - t[..., None])
        u = (1 - t).astype(np.float32)
        if rng.random() < 0.6:                                     # rain streaks
            rain = np.zeros((H, W), np.uint8)
            ang = rng.uniform(-20, 20)
            for _ in range(int(15 * sev)):
                x0, y0 = rng.integers(0, W), rng.integers(0, H)
                L = rng.integers(8, 20)
                x1 = int(x0 + L * np.sin(np.radians(ang))); y1 = int(y0 + L * np.cos(np.radians(ang)))
                cv2.line(rain, (int(x0), int(y0)), (x1, y1), 1, 1)
            rm = cv2.GaussianBlur(rain.astype(np.float32), (3, 3), 0)[..., None]
            out = out * (1 - 0.6 * rm) + 230 * 0.6 * rm
            u = np.maximum(u, 0.08 * sev * (rm[..., 0] > 0.05))
        return np.clip(out, 0, 255).astype(np.uint8), np.clip(u, 0, 1).astype(np.float32)

    # ------------------------------------------------------------------ compression / pixelation
    def compression(self, img, sev, rng):
        H, W = img.shape[:2]
        q = int(max(3, 42 - 8 * sev))
        f = 1.0 + 0.5 * sev
        small = cv2.resize(img, (max(8, int(W / f)), max(8, int(H / f))), interpolation=cv2.INTER_AREA)
        ok, buf = cv2.imencode(".jpg", cv2.cvtColor(small, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, q])
        dec = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        out = cv2.resize(dec, (W, H), interpolation=cv2.INTER_LINEAR)
        u = np.full((H, W), min(0.85, 0.10 + 0.12 * sev), np.float32)
        return out, u

    # ------------------------------------------------------------------ composition API
    def apply(self, img, type_name, sev, rng):
        fn = getattr(self, type_name)
        return fn(img, int(sev), rng)

    def random(self, img, rng, p_clean=0.15, max_types=2):
        """Random composition of 1-2 degradations. Returns img, u(HxW), type_map(HxW int)."""
        H, W = img.shape[:2]
        u_total = np.zeros((H, W), np.float32)
        type_map = np.zeros((H, W), np.int64)
        best_u = np.zeros((H, W), np.float32)
        if rng.random() < p_clean:
            return img, u_total, type_map
        names = DEG_TYPES[1:]
        k = 1 if rng.random() < 0.65 else max_types
        chosen = rng.choice(len(names), size=k, replace=False)
        out = img
        for c in chosen:
            name = names[c]
            sev = int(rng.integers(1, 6))
            out, u = self.apply(out, name, sev, rng)
            u_total = 1 - (1 - u_total) * (1 - u)
            upd = u > best_u
            type_map[upd] = TYPE_ID[name]
            best_u = np.maximum(best_u, u)
        type_map[u_total < 0.05] = 0
        return out, u_total, type_map

    def to_grid(self, u, type_map):
        """Pool pixel maps onto the token grid: mean unreliability + majority type (0 if reliable)."""
        g = self.grid
        H, W = u.shape
        u_grid = u.reshape(g, H // g, g, W // g).mean(axis=(1, 3)).astype(np.float32)
        t = type_map.reshape(g, H // g, g, W // g).transpose(0, 2, 1, 3).reshape(g, g, -1)
        t_grid = np.zeros((g, g), np.int64)
        for i in range(g):
            for j in range(g):
                cnt = np.bincount(t[i, j], minlength=len(DEG_TYPES))
                cnt[0] = 0
                t_grid[i, j] = int(cnt.argmax()) if cnt.max() > 0 and u_grid[i, j] >= 0.05 else 0
        return u_grid, t_grid


def visibility_from_unreliability(u, pts):
    """Per-landmark visibility target = 1 - u sampled (bilinear) at the landmark location."""
    H, W = u.shape
    x = np.clip(pts[:, 0], 0, W - 1.001); y = np.clip(pts[:, 1], 0, H - 1.001)
    x0, y0 = np.floor(x).astype(int), np.floor(y).astype(int)
    fx, fy = x - x0, y - y0
    v = (u[y0, x0] * (1 - fx) * (1 - fy) + u[y0, x0 + 1] * fx * (1 - fy)
         + u[y0 + 1, x0] * (1 - fx) * fy + u[y0 + 1, x0 + 1] * fx * fy)
    return (1.0 - v).astype(np.float32)

In [ ]:
%%writefile armor/data.py
"""WFLW loading, face-crop cache and the training / evaluation Dataset."""
import os
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset

from .common import (WFLW_FLIP, crop_affine, transform_points, warp, square_from_box,
                     render_landmark_heatmaps, render_edge_heatmaps, normalize_image, N_EDGES)
from .degrade import DegradationBank, visibility_from_unreliability, DEG_TYPES


# ----------------------------------------------------------------------------------------------
# Annotation parsing
# ----------------------------------------------------------------------------------------------
def parse_wflw_list(list_path):
    """Each line: 196 coords, 4 rect values (x_min y_min x_max y_max), 6 attribute flags, image name."""
    items = []
    with open(list_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 207:
                continue
            pts = np.array(p[:196], dtype=np.float32).reshape(98, 2)
            rect = np.array(p[196:200], dtype=np.float32)
            attrs = np.array(p[200:206], dtype=np.int64)
            items.append(dict(pts=pts, rect=rect, attrs=attrs, name=p[206]))
    return items


def wflw_lists(data_root):
    base = os.path.join(data_root, "WFLW_annotations", "list_98pt_rect_attr_train_test")
    return (os.path.join(base, "list_98pt_rect_attr_train.txt"),
            os.path.join(base, "list_98pt_rect_attr_test.txt"))


# ----------------------------------------------------------------------------------------------
# Face-crop cache (decode each JPEG once; training then runs from a uint8 numpy array)
# ----------------------------------------------------------------------------------------------
def build_cache(items, img_root, out_prefix, cache_size=160, cache_scale=1.6, limit=None, verbose=True):
    """Writes <out_prefix>_img.npy (N,S,S,3 uint8 RGB), _pts.npy (N,98,2 crop coords), _attrs.npy (N,6),
    _meta.npy (N,2,3 affine: crop coords -> original image coords)."""
    if limit:
        items = items[:limit]
    N = len(items)
    if os.path.exists(out_prefix + "_img.npy") and len(np.load(out_prefix + "_attrs.npy", mmap_mode="r")) == N:
        if verbose:
            print(f"cache exists: {out_prefix} ({N} faces)")
        return load_cache(out_prefix)
    imgs = np.lib.format.open_memmap(out_prefix + "_img.npy", mode="w+", dtype=np.uint8, shape=(N, cache_size, cache_size, 3))
    pts_all = np.zeros((N, 98, 2), np.float32)
    attrs = np.zeros((N, 6), np.int64)
    meta = np.zeros((N, 2, 3), np.float32)
    try:
        from tqdm.auto import tqdm
        it = tqdm(range(N), desc=f"caching {os.path.basename(out_prefix)}") if verbose else range(N)
    except ImportError:
        it = range(N)
    last_path, img = None, None
    for i in it:
        d = items[i]
        path = os.path.join(img_root, d["name"])
        if path != last_path:
            img = cv2.imread(path, cv2.IMREAD_COLOR)
            if img is None:
                raise FileNotFoundError(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            last_path = path
        # square around the annotation rect (same convention as detector boxes at inference)
        center, side = square_from_box(d["rect"], cache_scale)
        M = crop_affine(center, side, cache_size)
        imgs[i] = warp(img, M, cache_size)
        pts_all[i] = transform_points(d["pts"], M)
        attrs[i] = d["attrs"]
        meta[i] = cv2.invertAffineTransform(M)
    imgs.flush()
    np.save(out_prefix + "_pts.npy", pts_all)
    np.save(out_prefix + "_attrs.npy", attrs)
    np.save(out_prefix + "_meta.npy", meta)
    return load_cache(out_prefix)


def load_cache(out_prefix):
    return dict(img=np.load(out_prefix + "_img.npy", mmap_mode="r"),
                pts=np.load(out_prefix + "_pts.npy"),
                attrs=np.load(out_prefix + "_attrs.npy"),
                meta=np.load(out_prefix + "_meta.npy"))


# ----------------------------------------------------------------------------------------------
# Dataset
# ----------------------------------------------------------------------------------------------
class FaceDataset(Dataset):
    """Returns 128x128 crops (clean + degraded) with all supervision signals.

    train=True : random similarity augmentation + flip + colour jitter + degradation bank.
    train=False: deterministic centre crop; `fixed_deg=(type_name, severity)` turns the test set
                 into one cell of the WFLW-C benchmark (deterministic per index).
    """

    def __init__(self, cache, cfg, train=True, fixed_deg=None, return_clean_targets=True):
        self.c = cache
        self.cfg = cfg
        self.train = train
        self.fixed_deg = fixed_deg
        self.return_clean_targets = return_clean_targets
        self.S = cfg["crop_size"]
        self.H = cfg["heatmap_size"]
        self.bank = DegradationBank(grid=16)
        self.n = len(cache["attrs"])

    def __len__(self):
        return self.n

    # geometry of the crop inside the cached image --------------------------------------------
    def _affine(self, rng):
        cs, cache_size = self.cfg, self.cfg["cache_size"]
        # base crop = crop_scale * box, box = cache_size / cache_scale
        base_side = cache_size * cs["crop_scale"] / cs["cache_scale"]
        center = np.array([(cache_size - 1) / 2.0] * 2)
        rot, scale, flip = 0.0, 1.0, False
        if self.train:
            rot = float(rng.uniform(-cs["aug_rot"], cs["aug_rot"]))
            scale = float(1.0 + rng.uniform(-cs["aug_scale"], cs["aug_scale"]))
            center = center + rng.uniform(-cs["aug_shift"], cs["aug_shift"], 2) * base_side
            flip = bool(rng.random() < cs["aug_flip"])
        return crop_affine(center, base_side * scale, self.S, rot, flip), flip

    def _color_jitter(self, img, rng):
        a = self.cfg["aug_color"]
        if a <= 0:
            return img
        x = img.astype(np.float32)
        x = x * float(rng.uniform(1 - a, 1 + a)) + float(rng.uniform(-a, a)) * 64.0
        x = (x - x.mean()) * float(rng.uniform(1 - a, 1 + a)) + x.mean()
        return np.clip(x, 0, 255).astype(np.uint8)

    def __getitem__(self, i):
        rng = np.random.default_rng(None if self.train else (10_000_003 * i + 17))
        M, flip = self._affine(rng)
        img = warp(np.asarray(self.c["img"][i]), M, self.S)
        pts = transform_points(self.c["pts"][i], M)
        if flip:
            pts = pts[WFLW_FLIP]
        if self.train:
            img = self._color_jitter(img, rng)
        clean = img

        # ---- degradation bank -> unreliability / type / visibility targets
        u = np.zeros((self.S, self.S), np.float32)
        tmap = np.zeros((self.S, self.S), np.int64)
        deg = clean
        if self.fixed_deg is not None:
            name, sev = self.fixed_deg
            if name != "clean":
                deg, u = self.bank.apply(clean, name, sev, rng)
                tmap = np.where(u >= 0.05, DEG_TYPES.index(name), 0)
        elif self.train and rng.random() < self.cfg["p_degrade"]:
            deg, u, tmap = self.bank.random(clean, rng)
        u_grid, t_grid = self.bank.to_grid(u, tmap)
        vis = visibility_from_unreliability(u, pts)

        out = dict(
            img=torch.from_numpy(normalize_image(deg)),
            pts=torch.from_numpy(pts.astype(np.float32)),            # (98,2) in crop pixels
            unrel=torch.from_numpy(u_grid.reshape(-1)),             # (256,) alpha target
            dtype=torch.from_numpy(t_grid.reshape(-1)),             # (256,) type target
            vis=torch.from_numpy(vis),                              # (98,)
            attrs=torch.from_numpy(np.asarray(self.c["attrs"][i])),
            index=i,
        )
        if self.return_clean_targets:
            s = self.H / self.S
            out["img_clean"] = torch.from_numpy(normalize_image(clean))
            out["lmk_hm"] = torch.from_numpy(render_landmark_heatmaps(pts * s, self.H, self.cfg["lmk_sigma"]))
            out["edge_hm"] = torch.from_numpy(render_edge_heatmaps(pts * s, self.H, self.cfg["edge_sigma"]))
        return out


def make_loader(ds, bs, train, cfg):
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=train, drop_last=train,
                                       num_workers=cfg["num_workers"], pin_memory=True,
                                       persistent_workers=cfg["num_workers"] > 0)


# ----------------------------------------------------------------------------------------------
# Synthetic stand-in (lets the whole notebook run even before WFLW is downloaded)
# ----------------------------------------------------------------------------------------------
def make_synthetic_cache(n, cache_size=160, seed=0):
    """Cartoon faces with a deterministic 98-point layout: only for smoke-testing the pipeline."""
    rng = np.random.default_rng(seed)
    imgs = np.zeros((n, cache_size, cache_size, 3), np.uint8)
    pts = np.zeros((n, 98, 2), np.float32)
    base = _canonical_face_98()
    for i in range(n):
        s = cache_size * rng.uniform(0.45, 0.6)
        c = cache_size / 2 + rng.uniform(-6, 6, 2)
        ang = np.radians(rng.uniform(-15, 15))
        R = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
        p = (base - 0.5) @ R.T * s + c
        img = np.full((cache_size, cache_size, 3), rng.integers(40, 200, 3), np.uint8)
        skin = tuple(int(v) for v in rng.integers(120, 240, 3))
        cv2.fillPoly(img, [np.round(p[:33]).astype(np.int32)], skin)
        cv2.fillPoly(img, [np.round(np.vstack([p[:33], p[42:51][::-1]])).astype(np.int32)], skin)
        for grp in (p[33:42], p[42:51], p[60:68], p[68:76], p[76:88]):
            cv2.polylines(img, [np.round(grp).astype(np.int32)], True, (20, 20, 20), 2)
        cv2.polylines(img, [np.round(p[51:60]).astype(np.int32)], False, (60, 40, 40), 2)
        for q in (p[96], p[97]):
            cv2.circle(img, (int(q[0]), int(q[1])), 2, (0, 0, 0), -1)
        imgs[i] = img
        pts[i] = p
    attrs = rng.integers(0, 2, (n, 6)).astype(np.int64)
    meta = np.tile(np.array([[1, 0, 0], [0, 1, 0]], np.float32), (n, 1, 1))
    return dict(img=imgs, pts=pts, attrs=attrs, meta=meta)


def _canonical_face_98():
    """A plausible 98-point template in [0,1]^2 (used for the synthetic smoke-test data only)."""
    p = np.zeros((98, 2), np.float32)
    t = np.linspace(np.pi, 2 * np.pi, 33)                               # face contour (chin arc)
    p[:33, 0] = 0.5 + 0.42 * np.cos(t); p[:33, 1] = 0.45 - 0.5 * np.sin(t)
    def loop(idx, cx, cy, rx, ry):
        # WFLW order: start at the left-most point, run along the upper lid/edge to the right-most point,
        # then back along the lower edge (clockwise in image coordinates) - this makes the flip map consistent
        a = np.pi + np.linspace(0, 2 * np.pi, len(idx), endpoint=False)
        p[idx, 0] = cx + rx * np.cos(a); p[idx, 1] = cy + ry * np.sin(a)
    loop(list(range(33, 42)), 0.32, 0.30, 0.13, 0.03)                  # left brow
    loop(list(range(42, 51)), 0.68, 0.30, 0.13, 0.03)                  # right brow
    p[51:55] = np.stack([np.full(4, 0.5), np.linspace(0.40, 0.58, 4)], 1)   # nose bridge
    p[55:60] = np.stack([np.linspace(0.42, 0.58, 5), [0.62, 0.64, 0.65, 0.64, 0.62]], 1)
    loop(list(range(60, 68)), 0.34, 0.42, 0.08, 0.04)                  # left eye
    loop(list(range(68, 76)), 0.66, 0.42, 0.08, 0.04)                  # right eye
    a = np.linspace(np.pi, 0, 7); p[76:83] = np.stack([0.5 + 0.17 * np.cos(a), 0.78 - 0.04 * np.sin(a)], 1)
    a = np.linspace(0, np.pi, 7)[1:-1]; p[83:88] = np.stack([0.5 + 0.17 * np.cos(a), 0.78 + 0.07 * np.sin(a)], 1)
    a = np.linspace(np.pi, 0, 5); p[88:93] = np.stack([0.5 + 0.10 * np.cos(a), 0.785 - 0.01 * np.sin(a)], 1)
    a = np.linspace(0, np.pi, 5)[1:-1]; p[93:96] = np.stack([0.5 + 0.10 * np.cos(a), 0.785 + 0.03 * np.sin(a)], 1)
    p[96] = (0.34, 0.42); p[97] = (0.66, 0.42)
    return p

In [ ]:
%%writefile armor/vq.py
"""Stage A - quantized clean-face edge-heatmap generator (VQ prior)."""
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.body = nn.Sequential(nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv2d(ch, ch, 3, padding=1),
                                  nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv2d(ch, ch, 3, padding=1))

    def forward(self, x):
        return x + self.body(x)


class Encoder(nn.Module):
    """3x64x64 -> code_dim x16x16 (two stride-2 stages)."""

    def __init__(self, code_dim=128, ch=(64, 128), in_ch=3):
        super().__init__()
        c1, c2 = ch
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, c1, 3, padding=1), ResBlock(c1),
            nn.Conv2d(c1, c1, 4, stride=2, padding=1), ResBlock(c1),            # 32
            nn.Conv2d(c1, c2, 4, stride=2, padding=1), ResBlock(c2), ResBlock(c2),  # 16
            nn.GroupNorm(8, c2), nn.SiLU(), nn.Conv2d(c2, code_dim, 1))

    def forward(self, x):
        return self.net(x)


class Decoder(nn.Module):
    """code_dim x16x16 -> out_ch x64x64 edge heatmaps in [0,1]."""

    def __init__(self, code_dim=128, ch=(128, 64), out_ch=13):
        super().__init__()
        c1, c2 = ch
        self.net = nn.Sequential(
            nn.Conv2d(code_dim, c1, 3, padding=1), ResBlock(c1), ResBlock(c1),
            nn.ConvTranspose2d(c1, c2, 4, stride=2, padding=1), ResBlock(c2),      # 32
            nn.ConvTranspose2d(c2, c2, 4, stride=2, padding=1), ResBlock(c2),      # 64
            nn.GroupNorm(8, c2), nn.SiLU(), nn.Conv2d(c2, out_ch, 3, padding=1))

    def forward(self, z):
        return torch.sigmoid(self.net(z))


class VectorQuantizerEMA(nn.Module):
    """EMA codebook with dead-code restarts (stable even on short Colab schedules)."""

    def __init__(self, n_codes, dim, beta=0.25, decay=0.99, eps=1e-5, restart_after=200):
        super().__init__()
        self.n_codes, self.dim, self.beta, self.decay, self.eps = n_codes, dim, beta, decay, eps
        embed = torch.randn(n_codes, dim) * 0.1
        self.register_buffer("embed", embed)
        self.register_buffer("ema_count", torch.zeros(n_codes))
        self.register_buffer("ema_sum", embed.clone())
        self.register_buffer("unused", torch.zeros(n_codes))
        self.restart_after = restart_after

    @torch.no_grad()
    def nearest(self, flat):  # flat: (M, dim)
        d = (flat.pow(2).sum(1, keepdim=True) - 2 * flat @ self.embed.t() + self.embed.pow(2).sum(1)[None])
        return d.argmin(1)

    def lookup(self, idx):
        return F.embedding(idx, self.embed)

    def forward(self, z_e):
        """z_e: (B,D,H,W). Returns z_q (straight-through), idx (B,H,W), commitment loss, perplexity."""
        B, D, H, W = z_e.shape
        flat = z_e.permute(0, 2, 3, 1).reshape(-1, D).float()
        idx = self.nearest(flat)
        z_q = self.lookup(idx).view(B, H, W, D).permute(0, 3, 1, 2)
        loss = self.beta * F.mse_loss(z_e.float(), z_q.detach())
        if self.training:
            with torch.no_grad():
                onehot = F.one_hot(idx, self.n_codes).type(flat.dtype)
                count = onehot.sum(0)
                self.ema_count.mul_(self.decay).add_(count, alpha=1 - self.decay)
                self.ema_sum.mul_(self.decay).add_(onehot.t() @ flat, alpha=1 - self.decay)
                n = self.ema_count.sum()
                cnt = (self.ema_count + self.eps) / (n + self.n_codes * self.eps) * n
                self.embed.copy_(self.ema_sum / cnt.unsqueeze(1))
                # dead-code restart: codes unused for `restart_after` steps are re-seeded from the batch
                self.unused.add_(1.0); self.unused[count > 0] = 0
                dead = self.unused > self.restart_after
                if dead.any():
                    k = int(dead.sum())
                    pick = flat[torch.randint(0, flat.shape[0], (k,), device=flat.device)]
                    self.embed[dead] = pick
                    self.ema_sum[dead] = pick
                    self.ema_count[dead] = 1.0
                    self.unused[dead] = 0
        z_q_st = z_e + (z_q.to(z_e.dtype) - z_e).detach()
        probs = torch.bincount(idx, minlength=self.n_codes).float() / idx.numel()
        perplexity = torch.exp(-(probs * (probs + 1e-10).log()).sum())
        return z_q_st, idx.view(B, H, W), loss, perplexity


class HeatmapVQ(nn.Module):
    """Encoder E + codebook C + decoder D (ORFormer stage A, trained on *clean* faces)."""

    def __init__(self, cfg, n_edges=13):
        super().__init__()
        self.enc = Encoder(cfg["code_dim"], cfg["enc_ch"])
        self.vq = VectorQuantizerEMA(cfg["n_codes"], cfg["code_dim"], cfg["vq_beta"], cfg["vq_decay"])
        self.dec = Decoder(cfg["code_dim"], cfg["enc_ch"][::-1], n_edges)

    def forward(self, x64):
        z_e = self.enc(x64)
        z_q, idx, vq_loss, ppl = self.vq(z_e)
        hm = self.dec(z_q)
        return dict(hm=hm, idx=idx, vq_loss=vq_loss, perplexity=ppl, z_e=z_e, z_q=z_q)

    @torch.no_grad()
    def encode_indices(self, x64):
        z_e = self.enc(x64)
        B, D, H, W = z_e.shape
        return self.vq.nearest(z_e.permute(0, 2, 3, 1).reshape(-1, D).float()).view(B, H * W)

    @torch.no_grad()
    def quant_error(self, x64):
        """Per-image mean squared distance to the nearest code = VQ shift score (used by PG-TTA)."""
        z_e = self.enc(x64).float()
        B, D, H, W = z_e.shape
        flat = z_e.permute(0, 2, 3, 1).reshape(-1, D)
        idx = self.vq.nearest(flat)
        err = (flat - self.vq.lookup(idx)).pow(2).mean(1).view(B, -1).mean(1)
        return err

In [ ]:
%%writefile armor/orformer.py
"""Stage B - R-ORFormer: reliability- and degradation-type-supervised messenger-token recovery.

Differences from the original ORFormer (Chiang et al., WACV 2025):
  * the per-patch score alpha is supervised with the degradation bank's unreliability map
    (BCE), so it becomes a calibrated *reliability* estimate for any degradation, not only occlusion;
  * a degradation-type head predicts what corrupted each patch and its embedding conditions the
    messenger token before the next layer (type-conditioned recovery: a blurred patch still carries
    evidence, a fully occluded one does not);
  * reliability of the previous layer biases attention logits so unreliable patches stop contaminating
    the context that is used to recover them.
"""
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

from .vq import HeatmapVQ


class Attention(nn.Module):
    """Multi-head attention with an additive float mask broadcast as (B, 1|heads, L, S)."""

    def __init__(self, d, heads, drop):
        super().__init__()
        self.heads, self.dh = heads, d // heads
        self.q, self.kv, self.o = nn.Linear(d, d), nn.Linear(d, 2 * d), nn.Linear(d, d)
        self.drop = drop

    def forward(self, xq, xkv, mask=None):
        B, L, d = xq.shape
        S = xkv.shape[1]
        q = self.q(xq).view(B, L, self.heads, self.dh).transpose(1, 2)
        kv = self.kv(xkv).view(B, S, 2, self.heads, self.dh).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        if mask is not None:
            mask = mask.to(q.dtype)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=self.drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, L, d))


class ReliabilityLayer(nn.Module):
    def __init__(self, d, heads, ffn_mult, drop, n_types):
        super().__init__()
        self.ln_x1, self.ln_x2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ln_m1, self.ln_m2, self.ln_mx = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.self_attn = Attention(d, heads, drop)
        self.cross_attn = Attention(d, heads, drop)
        self.ffn_x = nn.Sequential(nn.Linear(d, d * ffn_mult), nn.GELU(), nn.Dropout(drop), nn.Linear(d * ffn_mult, d))
        self.ffn_m = nn.Sequential(nn.Linear(d, d * ffn_mult), nn.GELU(), nn.Dropout(drop), nn.Linear(d * ffn_mult, d))
        # reliability score from the disagreement between a patch and its messenger
        self.rel_head = nn.Sequential(nn.Linear(3 * d, d // 2), nn.GELU(), nn.Linear(d // 2, 1))
        self.type_head = nn.Sequential(nn.Linear(2 * d, d // 2), nn.GELU(), nn.Linear(d // 2, n_types))
        self.type_emb = nn.Linear(n_types, d, bias=False)

    def forward(self, X, M, alpha_prev, diag_mask):
        # alpha_prev (B,S): previous-layer unreliability -> additive key bias log(1 - alpha)
        bias = None if alpha_prev is None else torch.log1p(-alpha_prev.clamp(max=0.95))[:, None, None, :]
        h = self.ln_x1(X)
        X = X + self.self_attn(h, h, bias)                               # patch tokens, unreliable keys down-weighted
        X = X + self.ffn_x(self.ln_x2(X))
        mask = diag_mask if bias is None else (bias + diag_mask)         # messenger i never attends patch i
        M = M + self.cross_attn(self.ln_m1(M), self.ln_mx(X), mask)
        M = M + self.ffn_m(self.ln_m2(M))
        feat = torch.cat([X, M, (X - M).abs()], -1)
        alpha = torch.sigmoid(self.rel_head(feat).float()).squeeze(-1)   # (B,S) unreliability in [0,1]
        type_logits = self.type_head(torch.cat([X, M], -1))              # (B,S,K)
        M = M + self.type_emb(F.softmax(type_logits.float(), -1).to(M.dtype))  # type-conditioned recovery
        return X, M, alpha, type_logits


class RORFormer(nn.Module):
    def __init__(self, code_dim, n_codes, d=192, n_layers=3, heads=6, ffn_mult=4, drop=0.1, n_tokens=256, n_types=6):
        super().__init__()
        self.in_proj = nn.Linear(code_dim, d)
        self.pos = nn.Parameter(torch.randn(1, n_tokens, d) * 0.02)
        self.msg = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        self.layers = nn.ModuleList([ReliabilityLayer(d, heads, ffn_mult, drop, n_types) for _ in range(n_layers)])
        self.ln_x, self.ln_m = nn.LayerNorm(d), nn.LayerNorm(d)
        self.code_I = nn.Linear(d, n_codes)
        self.code_M = nn.Linear(d, n_codes)
        diag = torch.zeros(1, 1, n_tokens, n_tokens)
        diag[0, 0].fill_diagonal_(-1e4)
        self.register_buffer("diag_mask", diag)

    def forward(self, z_e):
        """z_e: (B,D,16,16) encoder features of the (possibly degraded) face."""
        B, D, H, W = z_e.shape
        X = self.in_proj(z_e.flatten(2).transpose(1, 2)) + self.pos
        M = self.msg.expand(B, H * W, -1) + self.pos
        alpha, alphas, types = None, [], []
        for layer in self.layers:
            X, M, alpha, tl = layer(X, M, alpha, self.diag_mask)
            alphas.append(alpha); types.append(tl)
        return dict(logits_I=self.code_I(self.ln_x(X)), logits_M=self.code_M(self.ln_m(M)),
                    alpha=alpha, alphas=alphas, type_logits=types[-1])


class ARMORPrior(nn.Module):
    """Encoder (fine-tuned) + R-ORFormer + frozen codebook & decoder -> degradation-free edge heatmaps."""

    def __init__(self, cfg, stage_a: HeatmapVQ, n_types=6):
        super().__init__()
        self.enc = copy.deepcopy(stage_a.enc)                              # fine-tuned in stage B
        self.enc_tgt = copy.deepcopy(stage_a.enc).requires_grad_(False)    # frozen: code targets + shift detector
        self.vq = copy.deepcopy(stage_a.vq).requires_grad_(False)
        self.dec = copy.deepcopy(stage_a.dec).requires_grad_(False)
        self.former = RORFormer(cfg["code_dim"], cfg["n_codes"], cfg["d_model"], cfg["n_layers"],
                                cfg["n_heads"], cfg["ffn_mult"], cfg["drop"], 256, n_types)
        self.n_codes = cfg["n_codes"]

    def train(self, mode=True):
        super().train(mode)
        self.enc_tgt.eval(); self.vq.eval(); self.dec.eval()                # frozen parts always in eval mode
        return self

    def _retrieve(self, logits):
        """Straight-through code retrieval: hard code features forward, gradient via the soft distribution."""
        soft = F.softmax(logits.float(), -1)
        soft_feat = soft @ self.vq.embed
        hard = self.vq.lookup(logits.argmax(-1))
        return hard + (soft_feat - soft_feat.detach())

    @torch.no_grad()
    def target_codes(self, x64_clean):
        z = self.enc_tgt(x64_clean)
        B, D, H, W = z.shape
        return self.vq.nearest(z.permute(0, 2, 3, 1).reshape(-1, D).float()).view(B, H * W)

    @torch.no_grad()
    def shift_score(self, x64):
        """VQ quantization error of the frozen clean-face encoder (large = far from the clean manifold)."""
        z = self.enc_tgt(x64).float()
        B, D, H, W = z.shape
        flat = z.permute(0, 2, 3, 1).reshape(-1, D)
        idx = self.vq.nearest(flat)
        return (flat - self.vq.lookup(idx)).pow(2).mean(1).view(B, -1).mean(1)

    def forward(self, x64):
        z_e = self.enc(x64)
        B, D, H, W = z_e.shape
        o = self.former(z_e)
        Z_I = self._retrieve(o["logits_I"])                                # (B,S,D) from the patch itself
        Z_M = self._retrieve(o["logits_M"])                                # (B,S,D) from the context
        a = o["alpha"].unsqueeze(-1)
        Z_rec = (1 - a) * Z_I + a * Z_M
        hm = self.dec(Z_rec.transpose(1, 2).reshape(B, D, H, W))
        o.update(hm=hm, z_rec=Z_rec)
        return o

In [ ]:
%%writefile armor/fld.py
"""Stage C - landmark network: lite stacked hourglass + reliability-gated prior fusion + visibility head."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from .orformer import ARMORPrior


# ---------------------------------------------------------------------------------- building blocks
class Bottleneck(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        mid = cout // 2
        self.body = nn.Sequential(
            nn.BatchNorm2d(cin), nn.ReLU(inplace=True), nn.Conv2d(cin, mid, 1, bias=False),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True), nn.Conv2d(mid, mid, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True), nn.Conv2d(mid, cout, 1, bias=False))
        self.skip = nn.Conv2d(cin, cout, 1, bias=False) if cin != cout else nn.Identity()

    def forward(self, x):
        return self.skip(x) + self.body(x)


class Hourglass(nn.Module):
    def __init__(self, depth, ch):
        super().__init__()
        self.depth = depth
        self.up1 = Bottleneck(ch, ch)
        self.low1 = Bottleneck(ch, ch)
        self.low2 = Hourglass(depth - 1, ch) if depth > 1 else Bottleneck(ch, ch)
        self.low3 = Bottleneck(ch, ch)

    def forward(self, x):
        up1 = self.up1(x)
        low = self.low3(self.low2(self.low1(F.max_pool2d(x, 2))))
        return up1 + F.interpolate(low, scale_factor=2, mode="nearest")


# ---------------------------------------------------------------------------------- soft-argmax + covariance
def heatmap_probs(hm, beta, window=9):
    """Windowed spatial softmax: softmax(beta * hm) restricted to a (window x window) box around the argmax.

    The window removes the centre bias of a global soft-argmax over 64x64 pixels while keeping the
    distribution differentiable; its covariance describes the local (anisotropic) shape of the peak.
    hm: (B,N,H,W) -> probs (B,N,H*W) float32
    """
    B, N, H, W = hm.shape
    flat = hm.float().flatten(2)
    if window and window < max(H, W):
        r = window // 2
        idx = flat.argmax(-1)
        ym, xm = idx // W, idx % W
        xs = torch.arange(W, device=hm.device); ys = torch.arange(H, device=hm.device)
        mx = (xs[None, None, :] - xm[..., None]).abs() <= r                    # (B,N,W)
        my = (ys[None, None, :] - ym[..., None]).abs() <= r                    # (B,N,H)
        mask = (my[..., :, None] & mx[..., None, :]).flatten(2)
        flat = flat.masked_fill(~mask, -1e4 / beta)
    return F.softmax(flat * beta, -1)


def softargmax_cov(hm, beta, window=9):
    """hm: (B,N,H,W) -> mu (B,N,2) [x,y] in heatmap pixels, cov (B,N,2,2), probs (B,N,H*W).

    The heatmap is turned into a distribution (see heatmap_probs); its mean is the landmark,
    its covariance feeds the STAR loss (anisotropic ambiguity) and the visibility head.
    """
    B, N, H, W = hm.shape
    p = heatmap_probs(hm, beta, window)
    xs = torch.arange(W, device=hm.device, dtype=torch.float32)
    ys = torch.arange(H, device=hm.device, dtype=torch.float32)
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    gx, gy = gx.reshape(-1), gy.reshape(-1)                                   # (HW,) pixel coordinates
    moments = p @ torch.stack([gx, gy, gx * gx, gy * gy, gx * gy], 1)          # (B,N,5): E[x],E[y],E[x2],E[y2],E[xy]
    mx, my, mxx, myy, mxy = moments.unbind(-1)
    cxx, cyy, cxy = (mxx - mx * mx).clamp(min=0), (myy - my * my).clamp(min=0), mxy - mx * my
    mu = torch.stack([mx, my], -1)                                            # (B,N,2)
    cov = torch.stack([torch.stack([cxx, cxy], -1), torch.stack([cxy, cyy], -1)], -2)  # (B,N,2,2)
    return mu, cov, p


# ---------------------------------------------------------------------------------- the network
class ARMORLandmarkNet(nn.Module):
    def __init__(self, cfg, prior: ARMORPrior, n_landmarks=98, n_edges=13):
        super().__init__()
        self.cfg, self.N, self.E = cfg, n_landmarks, n_edges
        ch, stacks = cfg["hg_ch"], cfg["hg_stacks"]
        self.prior = prior
        if not cfg["finetune_prior"]:
            self.prior.requires_grad_(False)
        # stem: 128 -> 64 (one stride-2 conv), feature width ch
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False), nn.BatchNorm2d(64),
                                  nn.ReLU(inplace=True), Bottleneck(64, ch))
        # reliability-gated fusion of the prior heatmaps
        self.prior_proj = nn.Sequential(nn.Conv2d(n_edges, ch, 1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(inplace=True))
        self.gate = nn.Conv2d(2 * ch, ch, 3, padding=1)
        nn.init.constant_(self.gate.bias, -2.0)                                   # gate starts almost closed (~0.12) ...
        self.kappa = nn.Parameter(torch.tensor(1.0))                              # ... and opens where the image is unreliable
        # hourglass stacks with landmark + edge heads and intermediate supervision
        self.hgs = nn.ModuleList([Hourglass(cfg["hg_depth"], ch) for _ in range(stacks)])
        self.feat = nn.ModuleList([nn.Sequential(Bottleneck(ch, ch), nn.Conv2d(ch, ch, 1, bias=False),
                                                 nn.BatchNorm2d(ch), nn.ReLU(inplace=True)) for _ in range(stacks)])
        self.out_lmk = nn.ModuleList([nn.Conv2d(ch, n_landmarks, 1) for _ in range(stacks)])
        self.out_edge = nn.ModuleList([nn.Conv2d(ch, n_edges, 1) for _ in range(stacks)])
        self.merge_f = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in range(stacks - 1)])
        self.merge_o = nn.ModuleList([nn.Conv2d(n_landmarks + n_edges, ch, 1) for _ in range(stacks - 1)])
        # per-landmark visibility head: sampled features + reliability + heatmap statistics
        self.vis_head = nn.Sequential(nn.Linear(ch + 4, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))

    # ------------------------------------------------------------------ helpers
    def run_prior(self, x):
        x64 = F.interpolate(x, size=64, mode="bilinear", align_corners=False)
        if self.cfg["finetune_prior"]:
            return self.prior(x64)
        with torch.no_grad():
            return self.prior(x64)

    def _sample(self, fmap, mu):
        """Bilinear sample fmap (B,C,H,W) at mu (B,N,2) given in fmap pixel coords -> (B,N,C)."""
        B, C, H, W = fmap.shape
        g = torch.stack([mu[..., 0] / (W - 1) * 2 - 1, mu[..., 1] / (H - 1) * 2 - 1], -1).unsqueeze(2)
        return F.grid_sample(fmap, g.to(fmap.dtype), align_corners=True).squeeze(-1).transpose(1, 2)

    # ------------------------------------------------------------------ forward
    def forward(self, x, prior_out=None):
        """x: (B,3,128,128) normalised crop. Returns dict with heatmaps, coords (input pixels), visibility."""
        B = x.shape[0]
        po = prior_out if prior_out is not None else self.run_prior(x)
        H_rec = po["hm"].to(x.dtype)                                              # (B,E,64,64)
        r = (1 - po["alpha"]).view(B, 1, 16, 16).to(x.dtype)
        r64 = F.interpolate(r, size=H_rec.shape[-1], mode="bilinear", align_corners=False)

        f = self.stem(x)                                                          # (B,ch,64,64)
        P = self.prior_proj(H_rec)
        g = torch.sigmoid(self.gate(torch.cat([f, P], 1)) + self.kappa * (1 - r64))
        f = f + g * P

        lmk_hms, edge_hms = [], []
        for i, hg in enumerate(self.hgs):
            y = self.feat[i](hg(f))
            lmk, edge = self.out_lmk[i](y), self.out_edge[i](y)
            lmk_hms.append(lmk); edge_hms.append(edge)
            if i < len(self.hgs) - 1:
                f = f + self.merge_f[i](y) + self.merge_o[i](torch.cat([lmk, edge], 1))

        mu, cov, probs = softargmax_cov(lmk_hms[-1], self.cfg["softargmax_beta"], self.cfg["softargmax_window"])   # heatmap pixels
        scale = x.shape[-1] / lmk_hms[-1].shape[-1]
        # visibility: features at the landmark, local reliability, peak value and spread
        fs = self._sample(y, mu)                                                  # (B,N,ch)
        rs = self._sample(r64, mu)                                                # (B,N,1)
        peak = lmk_hms[-1].flatten(2).amax(-1).unsqueeze(-1)                      # (B,N,1)
        tr = torch.log1p(cov[..., 0, 0] + cov[..., 1, 1]).unsqueeze(-1)                  # log spread
        ent = (-(probs * (probs + 1e-8).log()).sum(-1) / math.log(probs.shape[-1])).unsqueeze(-1)  # normalised entropy
        vis_logit = self.vis_head(torch.cat([fs.float(), rs.float(), peak.float(), tr, ent], -1).to(fs.dtype)).squeeze(-1)
        return dict(lmk_hms=lmk_hms, edge_hms=edge_hms, mu=mu, cov=cov, coords=mu * scale,
                    vis_logit=vis_logit, gate=g, H_rec=H_rec, alpha=po["alpha"], prior=po)

    # ------------------------------------------------------------------ parameter groups for PG-TTA
    def tta_parameters(self):
        """Normalisation affine params of the landmark branch + the fusion gate (the prior stays frozen)."""
        params = []
        for name, m in self.named_modules():
            if name == "prior" or name.startswith("prior."):      # the clean-face prior stays frozen
                continue
            if isinstance(m, (nn.BatchNorm2d, nn.GroupNorm, nn.LayerNorm)):
                params += [p for p in m.parameters(recurse=False)]
        params += list(self.gate.parameters()) + [self.kappa]
        return params

In [ ]:
%%writefile armor/losses.py
"""Losses: STAR-V (visibility-weighted STAR), Adaptive Wing, and the stage A/B/C objectives."""
import math
import torch
import torch.nn.functional as F


def eig2x2(cov, eps=1e-6):
    """Closed-form eigen-decomposition of symmetric 2x2 matrices (...,2,2) -> lam (...,2) desc, vecs (...,2,2)."""
    a, b, c = cov[..., 0, 0], cov[..., 0, 1], cov[..., 1, 1]
    tr, det = a + c, a * c - b * b
    disc = torch.sqrt(torch.clamp(tr * tr / 4 - det, min=0.0) + eps)
    l1, l2 = (tr / 2 + disc).clamp(min=0.0), (tr / 2 - disc).clamp(min=0.0)   # PSD up to rounding
    # eigenvector of l1: (b, l1 - a); for (near-)isotropic heatmaps fall back to the x axis.
    # NaN-safe: no .norm() at zero vectors (its gradient is NaN and leaks through torch.where).
    v1 = torch.stack([b, l1 - a], -1)
    n2 = (v1 * v1).sum(-1, keepdim=True)
    iso = n2 < 1e-10
    v1 = torch.where(iso, torch.tensor([1.0, 0.0], device=cov.device), v1 / torch.sqrt(n2 + 1e-12))
    v2 = torch.stack([-v1[..., 1], v1[..., 0]], -1)
    return torch.stack([l1, l2], -1), torch.stack([v1, v2], -2)   # vecs[..., k, :] = k-th eigenvector


def star_loss(mu, cov, gt, weight=None, detach_lambda=True, w_vr=0.0, eps=1e-4, detach_axes=True, lam_floor=0.25):
    """STAR loss (Zhou et al., CVPR 2023) with optional per-landmark weights (STAR-V).

    mu, gt: (B,N,2) in heatmap pixels; cov: (B,N,2,2) from the predicted heatmap distribution.
    The error is projected on the heatmap's principal axes and each component is divided by the
    corresponding sqrt(eigenvalue): errors along the ambiguous (long) axis are softened.
    """
    lam, vecs = eig2x2(cov.float())
    if detach_axes:                                                  # principal axes act as constants (stable)
        vecs = vecs.detach()
    e = (mu.float() - gt.float()).unsqueeze(-2)                      # (B,N,1,2)
    proj = (e * vecs).sum(-1)                                        # (B,N,2) components along v1, v2
    lam_d = lam.detach() if detach_lambda else lam
    lam_d = lam_d.clamp(min=lam_floor)                               # a one-hot peak must not blow the loss up
    per = F.smooth_l1_loss(proj / torch.sqrt(lam_d + eps), torch.zeros_like(proj), reduction="none").sum(-1)  # (B,N)
    if weight is not None:
        per = per * weight
    loss = per.mean()
    if w_vr > 0:                                                     # eigenvalue (value) restriction
        loss = loss + w_vr * lam.sum(-1).mean()
    return loss


def adaptive_wing_loss(pred, target, omega=14.0, theta=0.5, epsilon=1.0, alpha=2.1):
    """Adaptive Wing loss (Wang et al., ICCV 2019) for heatmap regression."""
    pred, target = pred.float(), target.float()
    diff = (target - pred).abs()
    A = omega * (1 / (1 + (theta / epsilon) ** (alpha - target))) * (alpha - target) * \
        ((theta / epsilon) ** (alpha - target - 1)) / epsilon
    C = theta * A - omega * torch.log1p((theta / epsilon) ** (alpha - target))
    small = diff < theta
    loss = torch.where(small, omega * torch.log1p((diff / epsilon) ** (alpha - target)), A * diff - C)
    # weight foreground pixels more (standard practice)
    w = 1 + 10 * (target > 0.2).float()
    return (loss * w).mean()


def _bce(p, t, eps=1e-5):
    """Plain BCE on probabilities (written out so it is safe inside torch.autocast)."""
    p = p.clamp(eps, 1 - eps)
    return -(t * p.log() + (1 - t) * (1 - p).log()).mean()


# ---------------------------------------------------------------------------------- stage objectives
def stage_a_loss(out, edge_hm):
    l_hm = F.mse_loss(out["hm"].float(), edge_hm.float())
    return l_hm + out["vq_loss"], dict(hm=l_hm.item(), vq=out["vq_loss"].item(), ppl=out["perplexity"].item())


def stage_b_loss(out, idx_tgt, edge_hm, unrel, dtype_tgt, cfg):
    """idx_tgt (B,256) codes of the clean face; unrel (B,256) in [0,1]; dtype_tgt (B,256) long."""
    B = idx_tgt.shape[0]
    l_code = F.cross_entropy(out["logits_I"].float().reshape(-1, cfg["n_codes"]), idx_tgt.reshape(-1)) + \
             F.cross_entropy(out["logits_M"].float().reshape(-1, cfg["n_codes"]), idx_tgt.reshape(-1))
    l_hm = F.mse_loss(out["hm"].float(), edge_hm.float())
    l_rel = sum(_bce(a.float(), unrel.float()) for a in out["alphas"]) / len(out["alphas"])
    l_type = F.cross_entropy(out["type_logits"].float().reshape(-1, out["type_logits"].shape[-1]), dtype_tgt.reshape(-1))
    total = cfg["w_code"] * l_code + cfg["w_hm"] * l_hm + cfg["w_rel"] * l_rel + cfg["w_type"] * l_type
    with torch.no_grad():
        acc_I = (out["logits_I"].argmax(-1) == idx_tgt).float().mean().item()
        rel_mae = (out["alpha"] - unrel).abs().mean().item()
    return total, dict(code=l_code.item(), hm=l_hm.item(), rel=l_rel.item(), type=l_type.item(), accI=acc_I, relMAE=rel_mae)


def stage_c_loss(out, batch, cfg, star_scale=1.0):
    """STAR-V on coordinates + AWing on heatmaps (intermediate supervision) + edge aux + visibility BCE.

    star_scale ramps STAR in during the first part of training (see cfg["star_warmup"]): AWing forms the
    peaks first, STAR-V then refines them sub-pixel; STAR's local window has nothing useful to say while
    the arg-max is still wrong."""
    s = out["lmk_hms"][-1].shape[-1] / batch["img"].shape[-1]
    gt_hm_coords = batch["pts"] * s
    vis_t = batch["vis"]
    w = cfg["vis_floor"] + (1 - cfg["vis_floor"]) * vis_t                 # never 0: GT exists under occlusion too
    gate = cfg.get("star_gate_px", 4)
    if gate:
        # STAR-V is a *refinement* loss: it only acts on landmarks whose heat-map peak already lies within
        # `gate` pixels of the target; wrong peaks are AWing's job (the windowed soft-argmax has no gradient there).
        with torch.no_grad():
            hm = out["lmk_hms"][-1]
            idx = hm.flatten(2).argmax(-1)
            am = torch.stack([idx % hm.shape[-1], idx // hm.shape[-1]], -1).float()
            near = ((am - gt_hm_coords).abs().amax(-1) <= gate).float()
        w = w * near
    l_star = star_loss(out["mu"], out["cov"], gt_hm_coords, w, cfg["star_detach"], cfg["star_vr"])
    l_aw = sum(adaptive_wing_loss(h, batch["lmk_hm"]) for h in out["lmk_hms"]) / len(out["lmk_hms"])
    l_edge = sum(F.mse_loss(e.float(), batch["edge_hm"].float()) for e in out["edge_hms"]) / len(out["edge_hms"])
    l_vis = F.binary_cross_entropy_with_logits(out["vis_logit"].float(), vis_t.float())
    total = star_scale * cfg["w_star"] * l_star + cfg["w_awing"] * l_aw + cfg["w_edge"] * l_edge + cfg["w_vis"] * l_vis
    return total, dict(star=l_star.item(), awing=l_aw.item(), edge=l_edge.item(), vis=l_vis.item())

In [ ]:
%%writefile armor/metrics.py
"""WFLW metrics: NME (inter-ocular), FR@0.10, AUC@0.10, per-attribute subsets."""
import numpy as np
from .common import LEFT_EYE_OUTER, RIGHT_EYE_OUTER, ATTR_NAMES


def nme_per_image(pred, gt):
    """pred, gt: (B,98,2) numpy. Returns (B,) NME normalised by the inter-ocular distance."""
    io = np.linalg.norm(gt[:, LEFT_EYE_OUTER] - gt[:, RIGHT_EYE_OUTER], axis=-1)
    err = np.linalg.norm(pred - gt, axis=-1).mean(-1)
    return err / np.maximum(io, 1e-6)


def failure_rate(nmes, thr=0.10):
    return float((nmes > thr).mean())


def auc_ced(nmes, thr=0.10, steps=1000):
    xs = np.linspace(0, thr, steps + 1)
    ced = np.array([(nmes <= x).mean() for x in xs])
    return float(np.trapezoid(ced, xs) / thr) if hasattr(np, "trapezoid") else float(np.trapz(ced, xs) / thr)


def summarize(nmes, attrs=None, thr=0.10):
    """Dict with full-set and per-subset NME (%), FR (%), AUC."""
    res = {"full": dict(NME=100 * nmes.mean(), FR=100 * failure_rate(nmes, thr), AUC=auc_ced(nmes, thr), n=len(nmes))}
    if attrs is not None:
        for j, name in enumerate(ATTR_NAMES):
            m = attrs[:, j] == 1
            if m.sum() > 0:
                res[name] = dict(NME=100 * nmes[m].mean(), FR=100 * failure_rate(nmes[m], thr), AUC=auc_ced(nmes[m], thr), n=int(m.sum()))
    return res


def format_table(res, title=""):
    lines = [title, f"{'subset':14s}{'n':>6s}{'NME%':>9s}{'FR%':>8s}{'AUC':>8s}"]
    for k, v in res.items():
        lines.append(f"{k:14s}{v['n']:6d}{v['NME']:9.3f}{v['FR']:8.2f}{v['AUC']:8.4f}")
    return "\n".join(lines)

In [ ]:
%%writefile armor/tta.py
"""Prior-Guided Test-Time Adaptation (PG-TTA) with a VQ shift detector.

Label-free, source-free adaptation of the landmark branch on the test crops themselves:
  L_prior : the backbone's edge head must agree with the frozen clean-face prior's reconstruction H_rec
  L_flip  : landmark heatmaps must be equivariant to horizontal flipping (with the 98-pt permutation)
  L_ent   : (small) entropy minimisation of the landmark heatmap distributions
Only normalisation affine parameters + the fusion gate are updated (see ARMORLandmarkNet.tta_parameters).

The number of gradient steps is *gated by the VQ shift score*: the quantisation error of the frozen
clean-face encoder measures how far a face lies from the clean manifold.  Clean faces get 0 steps
(no cost, no risk of degradation), heavily corrupted faces get up to `tta_max_steps`.
"""
import copy
import numpy as np
import torch
import torch.nn.functional as F

from .common import WFLW_FLIP
from .fld import heatmap_probs


class ShiftDetector:
    """Shift score of a test batch, z-scored against clean source crops.

    mode="alpha" (default): mean predicted unreliability of the R-ORFormer (its alpha head is trained on the
        degradation bank, so it is a calibrated, degradation-agnostic shift signal).
    mode="vq": quantisation error of the frozen clean-face VQ encoder. On WFLW this turned out to be a poor
        gate: blur / low-light / compression *remove* high-frequency detail and quantise as well as or better
        than clean faces (negative z), so only occlusion raised it. Kept for the ablation.
    """

    def __init__(self, prior, mode="alpha"):
        self.prior, self.mode = prior, mode
        self.mu, self.sigma = None, None

    @torch.no_grad()
    def score(self, x):
        x64 = F.interpolate(x, size=64, mode="bilinear", align_corners=False)
        if self.mode == "vq":
            return self.prior.shift_score(x64)
        was_training = self.prior.training
        self.prior.eval()
        s = self.prior(x64)["alpha"].mean(1)                                 # (B,) mean unreliability
        self.prior.train(was_training)
        return s

    @torch.no_grad()
    def fit(self, loader, device, max_batches=40):
        scores = []
        for i, b in enumerate(loader):
            scores.append(self.score(b["img_clean"].to(device, non_blocking=True)).cpu())
            if i + 1 >= max_batches:
                break
        s = torch.cat(scores)
        self.mu, self.sigma = float(s.mean()), float(s.std() + 1e-6)
        return self

    @torch.no_grad()
    def z(self, x):
        return (self.score(x) - self.mu) / self.sigma                      # (B,)

    def state_dict(self):
        return dict(mu=self.mu, sigma=self.sigma, mode=self.mode)

    def load_state_dict(self, d):
        self.mu, self.sigma, self.mode = d["mu"], d["sigma"], d.get("mode", self.mode)


def flip_heatmaps(hm):
    """Horizontally flip landmark heatmaps and permute channels so they align with the un-flipped face."""
    return hm.flip(-1)[:, WFLW_FLIP]


def tta_losses(model, x, out, cfg):
    losses = {}
    edge_pred = out["edge_hms"][-1].float()
    losses["prior"] = F.mse_loss(edge_pred, out["H_rec"].float().detach())
    if cfg["tta_w_flip"] > 0:
        out_f = model(x.flip(-1), prior_out=None)
        hm_f = flip_heatmaps(out_f["lmk_hms"][-1].float())
        losses["flip"] = F.mse_loss(out["lmk_hms"][-1].float(), hm_f)
    if cfg["tta_w_ent"] > 0:
        p = heatmap_probs(out["lmk_hms"][-1], cfg["softargmax_beta"], cfg["softargmax_window"])
        losses["ent"] = -(p * (p + 1e-8).log()).sum(-1).mean()
    total = cfg["tta_w_prior"] * losses["prior"] + cfg["tta_w_flip"] * losses.get("flip", 0) + cfg["tta_w_ent"] * losses.get("ent", 0)
    return total, losses


class PGTTA:
    """Episodic (default) or online prior-guided test-time adaptation wrapper."""

    def __init__(self, model, cfg, detector: ShiftDetector):
        self.model, self.cfg, self.det = model, cfg, detector
        self.params = model.tta_parameters()
        self._snapshot = [p.detach().clone() for p in self.params]
        self.opt = torch.optim.Adam(self.params, lr=cfg["tta_lr"])
        self.model.eval()
        for p in model.parameters():
            p.requires_grad_(False)
        for p in self.params:
            p.requires_grad_(True)

    def reset(self):
        with torch.no_grad():
            for p, s in zip(self.params, self._snapshot):
                p.copy_(s)
        self.opt = torch.optim.Adam(self.params, lr=self.cfg["tta_lr"])

    def steps_for(self, z):
        """Shift z-score -> number of adaptation steps (per batch: use the median)."""
        if self.cfg.get("tta_force_steps") is not None:          # ablation: no shift gating
            return int(self.cfg["tta_force_steps"])
        zb = float(torch.median(z))
        return int(np.clip(round(zb / self.cfg["tta_z_per_step"]), 0, self.cfg["tta_max_steps"]))

    def __call__(self, x):
        cfg = self.cfg
        z = self.det.z(x)
        n_steps = self.steps_for(z)
        if not cfg["tta_online"]:
            self.reset()
        for _ in range(n_steps):
            with torch.enable_grad():
                out = self.model(x)
                loss, _ = tta_losses(self.model, x, out, cfg)
                self.opt.zero_grad(set_to_none=True)
                loss.backward()
                self.opt.step()
        with torch.no_grad():
            out = self.model(x)
        out["tta_steps"] = n_steps
        out["shift_z"] = z
        return out

In [ ]:
%%writefile armor/detector.py
"""Stage 1 - SCRFD face detector (ONNX, InsightFace weights) + the full detect -> crop -> landmark -> remap pipeline."""
import os
import zipfile
import urllib.request
import numpy as np
import cv2
import torch
import torch.nn.functional as F

from .common import crop_affine, invert_affine, transform_points, warp, square_from_box, normalize_image

# InsightFace model packs (https://github.com/deepinsight/insightface/tree/master/python-package)
PACKS = {
    "det_500m": ("buffalo_sc", "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_sc.zip"),  # SCRFD-500M-KPS
    "det_10g": ("buffalo_l", "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"),     # SCRFD-10G-KPS
}


def download_scrfd(name="det_500m", root="/content/models"):
    """Returns the path to <name>.onnx, downloading the InsightFace pack if needed."""
    pack, url = PACKS[name]
    os.makedirs(root, exist_ok=True)
    onnx_path = os.path.join(root, f"{name}.onnx")
    if os.path.exists(onnx_path):
        return onnx_path
    zpath = os.path.join(root, f"{pack}.zip")
    if not os.path.exists(zpath):
        print(f"downloading {url} ...")
        urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as z:
        member = [m for m in z.namelist() if m.endswith(f"{name}.onnx")][0]
        with z.open(member) as src, open(onnx_path, "wb") as dst:
            dst.write(src.read())
    return onnx_path


class SCRFD:
    """Minimal re-implementation of InsightFace's SCRFD post-processing (strides 8/16/32, 2 anchors, 5 kps)."""

    def __init__(self, onnx_path, input_size=640, providers=None):
        import onnxruntime as ort
        if providers is None:
            avail = ort.get_available_providers()
            want = ["CUDAExecutionProvider", "CPUExecutionProvider"] if torch.cuda.is_available() else ["CPUExecutionProvider"]
            providers = [p for p in want if p in avail]
        self.session = ort.InferenceSession(onnx_path, providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]
        self.input_size = input_size
        self.strides = [8, 16, 32]
        self.num_anchors = 2
        self.use_kps = len(self.output_names) == 9
        self._centers = {}

    def _anchor_centers(self, h, w, stride):
        key = (h, w, stride)
        if key not in self._centers:
            c = np.stack(np.mgrid[:h, :w][::-1], axis=-1).astype(np.float32) * stride   # (h,w,2) as [x,y]
            c = c.reshape(-1, 2)
            c = np.stack([c] * self.num_anchors, axis=1).reshape(-1, 2)
            self._centers[key] = c
        return self._centers[key]

    @staticmethod
    def _nms(dets, thresh):
        x1, y1, x2, y2, s = dets.T
        areas = (x2 - x1 + 1) * (y2 - y1 + 1)
        order = s.argsort()[::-1]
        keep = []
        while order.size > 0:
            i = order[0]; keep.append(i)
            xx1 = np.maximum(x1[i], x1[order[1:]]); yy1 = np.maximum(y1[i], y1[order[1:]])
            xx2 = np.minimum(x2[i], x2[order[1:]]); yy2 = np.minimum(y2[i], y2[order[1:]])
            inter = np.maximum(0, xx2 - xx1 + 1) * np.maximum(0, yy2 - yy1 + 1)
            iou = inter / (areas[i] + areas[order[1:]] - inter)
            order = order[np.where(iou <= thresh)[0] + 1]
        return keep

    def detect(self, img_bgr, score_thresh=0.5, nms_thresh=0.4):
        """Returns boxes (N,5) [x1,y1,x2,y2,score] and kps (N,5,2) in original image coordinates."""
        S = self.input_size
        h0, w0 = img_bgr.shape[:2]
        scale = S / max(h0, w0)
        nh, nw = int(round(h0 * scale)), int(round(w0 * scale))
        resized = cv2.resize(img_bgr, (nw, nh))
        canvas = np.zeros((S, S, 3), np.uint8); canvas[:nh, :nw] = resized
        blob = cv2.dnn.blobFromImage(canvas, 1.0 / 128, (S, S), (127.5, 127.5, 127.5), swapRB=True)
        outs = self.session.run(self.output_names, {self.input_name: blob})
        fmc = len(self.strides)
        scores_l, boxes_l, kps_l = [], [], []
        for i, stride in enumerate(self.strides):
            scores = outs[i].reshape(-1)
            bbox = outs[i + fmc].reshape(-1, 4) * stride
            h, w = S // stride, S // stride
            ac = self._anchor_centers(h, w, stride)
            keep = np.where(scores >= score_thresh)[0]
            if keep.size == 0:
                continue
            ac, bbox, scores = ac[keep], bbox[keep], scores[keep]
            boxes = np.stack([ac[:, 0] - bbox[:, 0], ac[:, 1] - bbox[:, 1], ac[:, 0] + bbox[:, 2], ac[:, 1] + bbox[:, 3]], 1)
            scores_l.append(scores); boxes_l.append(boxes)
            if self.use_kps:
                kps = outs[i + 2 * fmc].reshape(-1, 5, 2)[keep] * stride + ac[:, None, :]
                kps_l.append(kps)
        if not scores_l:
            return np.zeros((0, 5), np.float32), np.zeros((0, 5, 2), np.float32)
        scores = np.concatenate(scores_l); boxes = np.concatenate(boxes_l) / scale
        dets = np.hstack([boxes, scores[:, None]]).astype(np.float32)
        order = scores.argsort()[::-1]
        dets = dets[order]
        keep = self._nms(dets, nms_thresh)
        kps = (np.concatenate(kps_l) / scale)[order][keep] if self.use_kps else np.zeros((len(keep), 5, 2), np.float32)
        return dets[keep], kps.astype(np.float32)


class FacePipeline:
    """SCRFD -> square crop (crop_scale x box) -> ARMOR landmark net -> landmarks back on the original image."""

    def __init__(self, detector: SCRFD, model, cfg, device, tta=None):
        self.det, self.model, self.cfg, self.device, self.tta = detector, model, cfg, device, tta
        self.model.eval()

    @torch.no_grad()
    def run(self, img_bgr, score_thresh=0.5, max_faces=32):
        boxes, _ = self.det.detect(img_bgr, score_thresh)
        boxes = boxes[:max_faces]
        if len(boxes) == 0:
            return []
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        S = self.cfg["crop_size"]
        crops, Ms = [], []
        for b in boxes:
            center, side = square_from_box(b, self.cfg["crop_scale"])
            M = crop_affine(center, side, S)
            crops.append(normalize_image(warp(rgb, M, S)))
            Ms.append(M)
        x = torch.from_numpy(np.stack(crops)).to(self.device)
        out = self.tta(x) if self.tta is not None else self.model(x)
        coords = out["coords"].float().cpu().numpy()
        vis = torch.sigmoid(out["vis_logit"].float()).cpu().numpy()
        alpha = out["alpha"].float().cpu().numpy().reshape(-1, 16, 16)
        results = []
        for i, b in enumerate(boxes):
            Minv = invert_affine(Ms[i])
            results.append(dict(box=b[:4], score=float(b[4]), landmarks=transform_points(coords[i], Minv),
                                visibility=vis[i], reliability=1 - alpha[i]))
        return results

    @staticmethod
    def draw(img_bgr, results, vis_thresh=0.5):
        out = img_bgr.copy()
        for r in results:
            x1, y1, x2, y2 = [int(v) for v in r["box"]]
            cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
            for (x, y), v in zip(r["landmarks"], r["visibility"]):
                color = (0, 255, 255) if v >= vis_thresh else (0, 0, 255)       # red = predicted non-visible
                cv2.circle(out, (int(round(x)), int(round(y))), 2, color, -1)
        return out


def roundtrip_test(n=200, seed=0):
    """Unit test: source -> crop -> source must be exact (<1e-3 px) for random similarity crops."""
    rng = np.random.default_rng(seed)
    worst = 0.0
    for _ in range(n):
        box = rng.uniform(0, 500, 2); box = np.concatenate([box, box + rng.uniform(20, 300, 2)])
        center, side = square_from_box(box, 1.3)
        M = crop_affine(center, side, 128, rng.uniform(-45, 45), bool(rng.random() < 0.5))
        pts = rng.uniform(-100, 700, (98, 2))
        back = transform_points(transform_points(pts, M), invert_affine(M))
        worst = max(worst, float(np.abs(back - pts).max()))
    return worst

In [ ]:
%%writefile armor/train.py
"""Training loops (stages A/B/C), evaluation, the WFLW-C benchmark and the PG-TTA benchmark."""
import os
import time
import json
import math
import numpy as np
import torch
import torch.nn.functional as F

from .vq import HeatmapVQ
from .orformer import ARMORPrior
from .fld import ARMORLandmarkNet
from .losses import stage_a_loss, stage_b_loss, stage_c_loss
from .metrics import nme_per_image, summarize, format_table
from .data import FaceDataset, make_loader
from .degrade import DEG_TYPES
from .tta import ShiftDetector, PGTTA
from .common import N_EDGES


# ---------------------------------------------------------------------------------- utils
def seed_all(seed):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def amp_ctx(cfg, device):
    use = cfg["amp"] and device.type == "cuda"
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use), use


def cosine_lr(opt, base_lr, step, total, warmup=100):
    warmup = max(1, min(warmup, total // 10))
    lr = base_lr * step / warmup if step < warmup else base_lr * 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))
    for g in opt.param_groups:
        g["lr"] = lr
    return lr


def save_ckpt(path, **objs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({k: (v.state_dict() if hasattr(v, "state_dict") else v) for k, v in objs.items()}, path)


def to_device(batch, device):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6


def x64(x):
    return F.interpolate(x, size=64, mode="bilinear", align_corners=False)


# ---------------------------------------------------------------------------------- stage A
def train_stage_a(cfg, train_cache, device, log=print):
    seed_all(cfg["seed"])
    ds = FaceDataset(train_cache, cfg, train=True)
    dl = make_loader(ds, cfg["a_bs"], True, cfg)
    model = HeatmapVQ(cfg, N_EDGES).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["a_lr"], weight_decay=1e-4)
    ctx, use_amp = amp_ctx(cfg, device)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    total = cfg["a_epochs"] * len(dl); step = 0; t0 = time.time()
    log(f"[A] HeatmapVQ {count_params(model):.2f}M params, {len(ds)} faces, {len(dl)} it/epoch")
    for ep in range(cfg["a_epochs"]):
        model.train()
        for b in dl:
            b = to_device(b, device)
            lr = cosine_lr(opt, cfg["a_lr"], step, total)
            with ctx:
                out = model(x64(b["img_clean"]))                  # stage A sees *clean* faces only
                loss, logs = stage_a_loss(out, b["edge_hm"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); step += 1
            if step % cfg["log_every"] == 0:
                log(f"[A] ep {ep} it {step}/{total} loss {loss.item():.4f} hm {logs['hm']:.4f} vq {logs['vq']:.4f} ppl {logs['ppl']:.1f} lr {lr:.2e} {time.time()-t0:.0f}s")
    save_ckpt(os.path.join(cfg["out_dir"], "stage_a.pt"), model=model, cfg=cfg)
    return model


# ---------------------------------------------------------------------------------- stage B
def train_stage_b(cfg, train_cache, stage_a, device, log=print):
    seed_all(cfg["seed"] + 1)
    ds = FaceDataset(train_cache, cfg, train=True)
    dl = make_loader(ds, cfg["b_bs"], True, cfg)
    prior = ARMORPrior(cfg, stage_a).to(device)
    params = [p for p in prior.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["b_lr"], weight_decay=1e-4)
    ctx, use_amp = amp_ctx(cfg, device)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    total = cfg["b_epochs"] * len(dl); step = 0; t0 = time.time()
    log(f"[B] R-ORFormer prior {count_params(prior.former):.2f}M (+enc {count_params(prior.enc):.2f}M) trainable")
    for ep in range(cfg["b_epochs"]):
        prior.train()
        for b in dl:
            b = to_device(b, device)
            lr = cosine_lr(opt, cfg["b_lr"], step, total)
            with ctx:
                idx_tgt = prior.target_codes(x64(b["img_clean"]))
                out = prior(x64(b["img"]))                        # degraded input
                loss, logs = stage_b_loss(out, idx_tgt, b["edge_hm"], b["unrel"], b["dtype"], cfg)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); step += 1
            if step % cfg["log_every"] == 0:
                log(f"[B] ep {ep} it {step}/{total} loss {loss.item():.3f} code {logs['code']:.3f} hm {logs['hm']:.4f} "
                    f"rel {logs['rel']:.3f} type {logs['type']:.3f} accI {logs['accI']:.3f} relMAE {logs['relMAE']:.3f} {time.time()-t0:.0f}s")
    save_ckpt(os.path.join(cfg["out_dir"], "stage_b.pt"), prior=prior, cfg=cfg)
    return prior


# ---------------------------------------------------------------------------------- stage C
def train_stage_c(cfg, train_cache, test_cache, prior, device, log=print, eval_every=1):
    seed_all(cfg["seed"] + 2)
    ds = FaceDataset(train_cache, cfg, train=True)
    dl = make_loader(ds, cfg["c_bs"], True, cfg)
    model = ARMORLandmarkNet(cfg, prior, 98, N_EDGES).to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["c_lr"], weight_decay=cfg["c_wd"])
    ctx, use_amp = amp_ctx(cfg, device)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    total = cfg["c_epochs"] * len(dl); step = 0; t0 = time.time(); best = 1e9
    log(f"[C] landmark net {count_params(model):.2f}M total, trainable {sum(p.numel() for p in params)/1e6:.2f}M")
    for ep in range(cfg["c_epochs"]):
        model.train()
        if not cfg["finetune_prior"]:
            model.prior.eval()
        for b in dl:
            b = to_device(b, device)
            lr = cosine_lr(opt, cfg["c_lr"], step, total, warmup=200)
            star_scale = min(1.0, step / max(1.0, cfg["star_warmup"] * total))
            with ctx:
                out = model(b["img"])
                loss, logs = stage_c_loss(out, b, cfg, star_scale)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 5.0)
            scaler.step(opt); scaler.update(); step += 1
            if step % cfg["log_every"] == 0:
                log(f"[C] ep {ep} it {step}/{total} loss {loss.item():.3f} star {logs['star']:.3f} awing {logs['awing']:.3f} "
                    f"edge {logs['edge']:.4f} vis {logs['vis']:.3f} lr {lr:.2e} {time.time()-t0:.0f}s")
        if (ep + 1) % eval_every == 0 or ep == cfg["c_epochs"] - 1:
            res = evaluate(cfg, model, test_cache, device)
            log(f"[C] epoch {ep}: test NME {res['full']['NME']:.3f}%  FR {res['full']['FR']:.2f}%  AUC {res['full']['AUC']:.4f}")
            if res["full"]["NME"] < best:
                best = res["full"]["NME"]
                save_ckpt(os.path.join(cfg["out_dir"], "stage_c_best.pt"), model=model, cfg=cfg)
    save_ckpt(os.path.join(cfg["out_dir"], "stage_c_last.pt"), model=model, cfg=cfg)
    return model


# ---------------------------------------------------------------------------------- evaluation
@torch.no_grad()
def predict(cfg, model, loader, device, tta=None):
    model.eval()
    preds, gts, attrs, vis_p, vis_t, steps, zs = [], [], [], [], [], [], []
    ctx, _ = amp_ctx(cfg, device)
    for b in loader:
        b = to_device(b, device)
        if tta is not None:
            out = tta(b["img"]); steps.append(out["tta_steps"]); zs.append(float(out["shift_z"].median()))
        else:
            with ctx:
                out = model(b["img"])
        preds.append(out["coords"].float().cpu().numpy()); gts.append(b["pts"].cpu().numpy())
        attrs.append(b["attrs"].cpu().numpy())
        vis_p.append(torch.sigmoid(out["vis_logit"].float()).cpu().numpy()); vis_t.append(b["vis"].cpu().numpy())
    return dict(pred=np.concatenate(preds), gt=np.concatenate(gts), attrs=np.concatenate(attrs),
                vis_p=np.concatenate(vis_p), vis_t=np.concatenate(vis_t), steps=steps, zs=zs)


def evaluate(cfg, model, test_cache, device, fixed_deg=None, tta=None, verbose=False):
    ds = FaceDataset(test_cache, cfg, train=False, fixed_deg=fixed_deg, return_clean_targets=False)
    dl = make_loader(ds, 64, False, cfg)
    P = predict(cfg, model, dl, device, tta)
    nmes = nme_per_image(P["pred"], P["gt"])
    res = summarize(nmes, P["attrs"])
    vis_auc = _vis_accuracy(P["vis_p"], P["vis_t"])
    res["full"]["vis_acc"] = vis_auc
    if P["steps"]:
        res["full"]["tta_steps"] = float(np.mean(P["steps"]))
        res["full"]["tta_z"] = float(np.median(P["zs"]))
    if verbose:
        print(format_table(res, title=f"WFLW test {fixed_deg or ''}"))
    return res


def _vis_accuracy(p, t, thr=0.5):
    """Balanced accuracy of visibility prediction (only meaningful on degraded test cells)."""
    tb = t >= thr; pb = p >= thr
    if tb.all() or (~tb).all():
        return float((pb == tb).mean())
    return 0.5 * (float((pb & tb).sum() / tb.sum()) + float((~pb & ~tb).sum() / (~tb).sum()))


# ---------------------------------------------------------------------------------- WFLW-C benchmark
def wflw_c_benchmark(cfg, model, test_cache, device, severities=(1, 3, 5), types=None, tta=None, log=print):
    """NME on every (degradation type, severity) cell of the test set.  Returns {type: {sev: NME%}}."""
    types = types or DEG_TYPES[1:]
    table = {"clean": {0: evaluate(cfg, model, test_cache, device, None, tta)["full"]["NME"]}}
    log(f"clean: NME {table['clean'][0]:.3f}%")
    for t in types:
        table[t] = {}
        for s in severities:
            r = evaluate(cfg, model, test_cache, device, (t, s), tta)
            table[t][s] = r["full"]["NME"]
            log(f"{t:12s} sev {s}: NME {r['full']['NME']:.3f}%  FR {r['full']['FR']:.2f}%  vis-acc {r['full']['vis_acc']:.3f}"
                + (f"  TTA steps {r['full']['tta_steps']:.2f}" if 'tta_steps' in r['full'] else ""))
    mean_c = float(np.mean([v for t in types for v in table[t].values()]))
    table["mean_corrupted"] = {0: mean_c}
    log(f"mean corrupted NME {mean_c:.3f}%")
    return table


def build_tta(cfg, model, train_cache, device, shift_mode="alpha"):
    det = ShiftDetector(model.prior, mode=shift_mode)
    calib = FaceDataset(train_cache, cfg, train=False)                 # clean centre crops for calibration
    det.fit(make_loader(calib, 64, False, cfg), device)
    return PGTTA(model, cfg, det), det


# ---------------------------------------------------------------------------------- loading
def load_stage_a(cfg, device):
    ck = torch.load(os.path.join(cfg["out_dir"], "stage_a.pt"), map_location=device, weights_only=False)
    m = HeatmapVQ(ck["cfg"], N_EDGES).to(device); m.load_state_dict(ck["model"]); return m.eval()


def load_prior(cfg, stage_a, device):
    ck = torch.load(os.path.join(cfg["out_dir"], "stage_b.pt"), map_location=device, weights_only=False)
    p = ARMORPrior(ck["cfg"], stage_a).to(device); p.load_state_dict(ck["prior"]); return p.eval()


def load_landmark_net(cfg, prior, device, which="best"):
    ck = torch.load(os.path.join(cfg["out_dir"], f"stage_c_{which}.pt"), map_location=device, weights_only=False)
    m = ARMORLandmarkNet(ck["cfg"], prior, 98, N_EDGES).to(device); m.load_state_dict(ck["model"]); return m.eval()

In [ ]:
import importlib, sys
sys.path.insert(0, WORK)
for name in list(sys.modules):
    if name.startswith("armor"):
        del sys.modules[name]
import armor
from armor.config import get_config
from armor.common import N_EDGES, ATTR_NAMES, denormalize_image
from armor.data import parse_wflw_list, wflw_lists, build_cache, load_cache, FaceDataset, make_loader, make_synthetic_cache
from armor.degrade import DegradationBank, DEG_TYPES
from armor import train as T
from armor.detector import roundtrip_test
print("armor", armor.__version__, "| affine round-trip worst error (px):", roundtrip_test())

## 2 · Configuration

In [ ]:
MODE = "QUICK"           # "QUICK" (sanity, ~20 min on T4) or "FULL" (real schedule)
USE_SYNTHETIC = False    # True -> skip WFLW and run on synthetic faces (code-path validation only)
SCRFD_MODEL = "det_500m" # "det_500m" (SCRFD-500M, fastest) or "det_10g" (SCRFD-10G, more accurate)
WFLW_DRIVE_DIR = "/content/drive/MyDrive/WFLW"   # folder on your Drive holding WFLW_images.tar.gz (or the extracted WFLW_images/ and WFLW_annotations/)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = get_config(MODE, out_dir=OUT_DIR, data_root=f"{WORK}/data/WFLW", cache_dir=f"{WORK}/cache",
                 num_workers=2 if device.type == "cuda" else 0, amp=device.type == "cuda")
if device.type == "cpu":   # keep a CPU run tractable
    cfg.update(train_subset=200, test_subset=60, a_epochs=1, b_epochs=1, c_epochs=1, hg_ch=64)
SEVERITIES = (1, 3, 5) if MODE == "FULL" else (3,)
print(json.dumps({k: v for k, v in cfg.items() if k in ["train_subset","a_epochs","b_epochs","c_epochs","n_codes","code_dim","crop_size","hg_ch","hg_stacks"]}, indent=1))
print("device:", device)

## 3 · Data — WFLW download, face-crop cache, degradation bank

In [ ]:
DATA = cfg["data_root"]; os.makedirs(DATA, exist_ok=True)
IMG_TAR, ANN_TAR = f"{DATA}/WFLW_images.tar.gz", f"{DATA}/WFLW_annotations.tar.gz"
GDRIVE_ID = "1hzBd48JIdWTJSsATBEB_eFVvPL1bx6UC"                     # official WFLW images (≈725 MB)
ANN_URLS = ["https://wywu.github.io/projects/LAB/support/WFLW_annotations.tar.gz",                      # official (404 since 2026)
            "https://web.archive.org/web/20260103234337id_/https://wywu.github.io/projects/LAB/support/WFLW_annotations.tar.gz"]  # Internet Archive copy

def sh(cmd):
    print("$", cmd); return subprocess.run(cmd, shell=True).returncode

if not USE_SYNTHETIC:
    if os.path.isdir(WFLW_DRIVE_DIR):
        print("your Drive folder contains:", sorted(os.listdir(WFLW_DRIVE_DIR)))
        # already-extracted folders on Drive -> copy them to the fast local disk
        for sub in ["WFLW_images", "WFLW_annotations"]:
            if os.path.isdir(f"{WFLW_DRIVE_DIR}/{sub}") and not os.path.isdir(f"{DATA}/{sub}"):
                sh(f"cp -r '{WFLW_DRIVE_DIR}/{sub}' '{DATA}/'")
        if os.path.exists(f"{WFLW_DRIVE_DIR}/WFLW_annotations.tar.gz") and not os.path.isdir(f"{DATA}/WFLW_annotations"):
            sh(f"tar -xzf '{WFLW_DRIVE_DIR}/WFLW_annotations.tar.gz' -C {DATA}")
    if not os.path.isdir(f"{DATA}/WFLW_images"):
        cands = [IMG_TAR, f"{WFLW_DRIVE_DIR}/WFLW_images.tar.gz", "/content/drive/MyDrive/WFLW/WFLW_images.tar.gz", f"{OUT_DIR}/WFLW_images.tar.gz"]
        if os.path.isdir(WFLW_DRIVE_DIR):   # any *.tar.gz / *.tgz in the folder, whatever Drive renamed it to
            cands += [os.path.join(WFLW_DRIVE_DIR, f) for f in os.listdir(WFLW_DRIVE_DIR) if f.startswith("WFLW_images") and (f.endswith(".gz") or f.endswith(".tgz"))]
        for cand in cands:
            if os.path.exists(cand) and os.path.getsize(cand) > 7e8:
                IMG_TAR = cand; break
        else:
            sh(f"gdown 'https://drive.google.com/uc?id={GDRIVE_ID}' -O {IMG_TAR}")
        if os.path.exists(IMG_TAR) and os.path.getsize(IMG_TAR) > 7e8:
            print("extracting", IMG_TAR, "(a few minutes from Drive) ...")
            sh(f"tar -xzf '{IMG_TAR}' -C {DATA}")
        else:
            print("WFLW_images.tar.gz not found / incomplete; looked in:", cands)
    ann_list = f"{DATA}/WFLW_annotations/list_98pt_rect_attr_train_test/list_98pt_rect_attr_train.txt"
    if not os.path.exists(ann_list):
        import urllib.request, glob, shutil
        for src in ANN_URLS:
            try:
                req = urllib.request.Request(src, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=180) as r, open(ANN_TAR, "wb") as f:
                    f.write(r.read())
                sh(f"tar -xzf '{ANN_TAR}' -C {DATA}")
                if os.path.exists(ann_list):
                    print("annotations from", src); break
            except Exception as e:
                print("annotation source failed:", src, "->", repr(e)[:100])
        if not os.path.exists(ann_list):   # tolerate a different folder layout inside the archive
            for f in glob.glob(f"{DATA}/WFLW_annotations/**/list_98pt_rect_attr_*.txt", recursive=True):
                os.makedirs(os.path.dirname(ann_list), exist_ok=True); shutil.copy(f, os.path.dirname(ann_list))
    ok = os.path.isdir(f"{DATA}/WFLW_images") and os.path.exists(ann_list)
    if not ok:
        print("\n*** WFLW not available: images present =", os.path.isdir(f"{DATA}/WFLW_images"), "| annotations present =", os.path.exists(ann_list),
              "\n    Put WFLW_images.tar.gz (and, if the download above failed, WFLW_annotations.tar.gz) in WFLW_DRIVE_DIR and re-run this cell. ***")
        USE_SYNTHETIC = True
# synthetic runs live in their own folder so their checkpoints can never be mistaken for WFLW ones
cfg["out_dir"] = os.path.join(OUT_DIR, "synthetic" if USE_SYNTHETIC else MODE.lower())
os.makedirs(cfg["out_dir"], exist_ok=True)
print("synthetic mode:", USE_SYNTHETIC, "| checkpoints ->", cfg["out_dir"])
if USE_SYNTHETIC:
    print("*** This is only a code check. For real results put WFLW_images.tar.gz in MyDrive/WFLW/ and re-run this cell. ***")

In [ ]:
os.makedirs(cfg["cache_dir"], exist_ok=True)
if USE_SYNTHETIC:
    train_cache = make_synthetic_cache(cfg["train_subset"] or 2000, cfg["cache_size"], seed=0)
    test_cache = make_synthetic_cache(cfg["test_subset"] or 500, cfg["cache_size"], seed=1)
else:
    tr_list, te_list = wflw_lists(DATA)
    tr_items, te_items = parse_wflw_list(tr_list), parse_wflw_list(te_list)
    print(f"WFLW: {len(tr_items)} train / {len(te_items)} test faces")
    img_root = f"{DATA}/WFLW_images"
    tag = f"s{cfg['cache_size']}_{cfg['train_subset'] or 'all'}"
    # reuse caches saved on Drive from a previous session (fast), else build and back them up
    for split in ["train", "test"]:
        for suf in ["_img.npy", "_pts.npy", "_attrs.npy", "_meta.npy"]:
            src = f"{OUT_DIR}/cache/{split}_{tag}{suf}"; dst = f"{cfg['cache_dir']}/{split}_{tag}{suf}"
            if os.path.exists(src) and not os.path.exists(dst):
                sh(f"cp '{src}' '{dst}'")
    train_cache = build_cache(tr_items, img_root, f"{cfg['cache_dir']}/train_{tag}", cfg["cache_size"], cfg["cache_scale"], cfg["train_subset"])
    test_cache = build_cache(te_items, img_root, f"{cfg['cache_dir']}/test_{tag}", cfg["cache_size"], cfg["cache_scale"], cfg["test_subset"])
    if USE_DRIVE and IN_COLAB:
        os.makedirs(f"{OUT_DIR}/cache", exist_ok=True)
        sh(f"cp -n {cfg['cache_dir']}/*_{tag}_*.npy '{OUT_DIR}/cache/' 2>/dev/null")
print("train faces:", len(train_cache["attrs"]), "| test faces:", len(test_cache["attrs"]))
print("test attribute counts:", dict(zip(ATTR_NAMES, test_cache["attrs"].sum(0).tolist())))

In [ ]:
# ---- visual sanity: augmented crops with the degradation bank, alpha/type targets and visibility targets
import numpy as np, cv2, matplotlib.pyplot as plt
ds_vis = FaceDataset(train_cache, cfg, train=True)
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for j in range(6):
    it = ds_vis[j]
    img = denormalize_image(it["img"].numpy()).copy()
    for (x, y), v in zip(it["pts"].numpy(), it["vis"].numpy()):
        cv2.circle(img, (int(x), int(y)), 1, (0, 255, 0) if v > 0.5 else (255, 0, 0), -1)
    axes[0, j].imshow(img); axes[0, j].set_title("degraded crop + GT (red = not visible)", fontsize=8)
    axes[1, j].imshow(it["unrel"].view(16, 16), vmin=0, vmax=1, cmap="magma"); axes[1, j].set_title("unreliability target (alpha)", fontsize=8)
    axes[2, j].imshow(it["dtype"].view(16, 16), vmin=0, vmax=5, cmap="tab10"); axes[2, j].set_title("degradation type target", fontsize=8)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

bank = DegradationBank(); rng = np.random.default_rng(0)
base = np.asarray(train_cache["img"][0])[16:144, 16:144]
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
axes[0, 0].imshow(base); axes[0, 0].set_title("clean"); axes[1, 0].axis("off")
for j, name in enumerate(DEG_TYPES[1:], 1):
    d, u = bank.apply(base, name, 4, rng)
    axes[0, j].imshow(d); axes[0, j].set_title(f"{name} (sev 4)"); axes[1, j].imshow(u, vmin=0, vmax=1, cmap="magma"); axes[1, j].set_title("unreliability")
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 4 · Stage A — clean-face VQ prior (quantized edge-heatmap generator)

In [ ]:
if os.path.exists(f"{cfg['out_dir']}/stage_a.pt"):
    print("loading stage A checkpoint from", cfg["out_dir"]); stage_a = T.load_stage_a(cfg, device)
else:
    stage_a = T.train_stage_a(cfg, train_cache, device)

In [ ]:
# reconstructions on held-out clean faces
import torch.nn.functional as F
ds_te_clean = FaceDataset(test_cache, cfg, train=False)
b = next(iter(make_loader(ds_te_clean, 6, False, cfg)))
stage_a.eval()
with torch.no_grad():
    o = stage_a(F.interpolate(b["img_clean"].to(device), size=64, mode="bilinear", align_corners=False))
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for j in range(6):
    axes[0, j].imshow(denormalize_image(b["img_clean"][j].numpy())); axes[0, j].set_title("clean face")
    axes[1, j].imshow(b["edge_hm"][j].max(0)[0], cmap="gray"); axes[1, j].set_title("GT edge heatmaps")
    axes[2, j].imshow(o["hm"][j].max(0)[0].float().cpu(), cmap="gray"); axes[2, j].set_title(f"VQ reconstruction (ppl {float(o['perplexity']):.0f})")
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 5 · Stage B — R-ORFormer: reliability-supervised, type-conditioned messenger recovery

In [ ]:
if os.path.exists(f"{cfg['out_dir']}/stage_b.pt"):
    print("loading stage B checkpoint from", cfg["out_dir"]); prior = T.load_prior(cfg, stage_a, device)
else:
    prior = T.train_stage_b(cfg, train_cache, stage_a, device)

In [ ]:
# degraded input -> predicted unreliability alpha, degradation type, recovered heatmaps
ds_occ = FaceDataset(test_cache, cfg, train=False, fixed_deg=("occlusion", 4))
ds_blur = FaceDataset(test_cache, cfg, train=False, fixed_deg=("blur", 4))
prior.eval()
fig, axes = plt.subplots(5, 6, figsize=(18, 15))
for j in range(6):
    it = (ds_occ if j < 3 else ds_blur)[j]
    with torch.no_grad():
        o = prior(F.interpolate(it["img"][None].to(device), size=64, mode="bilinear", align_corners=False))
    axes[0, j].imshow(denormalize_image(it["img"].numpy())); axes[0, j].set_title("degraded input")
    axes[1, j].imshow(it["unrel"].view(16, 16), vmin=0, vmax=1, cmap="magma"); axes[1, j].set_title("unreliability target")
    axes[2, j].imshow(o["alpha"][0].view(16, 16).float().cpu(), vmin=0, vmax=1, cmap="magma"); axes[2, j].set_title("predicted alpha")
    axes[3, j].imshow(o["type_logits"][0].argmax(-1).view(16, 16).cpu(), vmin=0, vmax=5, cmap="tab10"); axes[3, j].set_title("predicted type")
    axes[4, j].imshow(o["hm"][0].max(0)[0].float().cpu(), cmap="gray"); axes[4, j].set_title("recovered edge heatmaps")
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 6 · Stage C — landmark network with reliability-gated fusion, visibility head and STAR-V

In [ ]:
if os.path.exists(f"{cfg['out_dir']}/stage_c_best.pt"):
    print("loading stage C checkpoint from", cfg["out_dir"]); model = T.load_landmark_net(cfg, prior, device, "best")
else:
    model = T.train_stage_c(cfg, train_cache, test_cache, prior, device, eval_every=max(1, cfg["c_epochs"] // 6))

In [ ]:
from armor.metrics import format_table
res_clean = T.evaluate(cfg, model, test_cache, device, verbose=True)
RESULTS = {"mode": MODE, "synthetic": USE_SYNTHETIC, "wflw_test": res_clean}

In [ ]:
# qualitative results under the WFLW-C degradations (yellow = visible, red = predicted non-visible)
def show_predictions(model, cache, degs, n=6, tta=None):
    fig, axes = plt.subplots(len(degs), n, figsize=(3 * n, 3 * len(degs)))
    axes = np.atleast_2d(axes)
    for r, deg in enumerate(degs):
        ds = FaceDataset(cache, cfg, train=False, fixed_deg=deg, return_clean_targets=False)
        idx = np.linspace(0, len(ds) - 1, n).astype(int)
        batch = torch.stack([ds[i]["img"] for i in idx]).to(device)
        with torch.no_grad():
            out = tta(batch) if tta is not None else model(batch)
        coords = out["coords"].float().cpu().numpy(); vis = torch.sigmoid(out["vis_logit"].float()).cpu().numpy()
        for c in range(n):
            img = denormalize_image(batch[c].cpu().numpy()).copy()
            for (x, y), v in zip(coords[c], vis[c]):
                cv2.circle(img, (int(round(x)), int(round(y))), 1, (255, 255, 0) if v > 0.5 else (255, 0, 0), -1)
            axes[r, c].imshow(img); axes[r, c].axis("off")
        axes[r, 0].set_title(str(deg or "clean"), loc="left")
    plt.tight_layout(); plt.show()
show_predictions(model, test_cache, [None, ("occlusion", 4), ("blur", 4), ("lowlight", 4), ("weather", 4)])

## 7 · WFLW-C benchmark (no adaptation)

In [ ]:
wflw_c = T.wflw_c_benchmark(cfg, model, test_cache, device, severities=SEVERITIES)
RESULTS["wflw_c"] = wflw_c

## 8 · Prior-Guided Test-Time Adaptation (PG-TTA)
The shift detector is calibrated on clean training crops; each test batch's VQ quantisation error is converted to a z-score,
which decides how many adaptation steps (0 … `tta_max_steps`) to spend. Only normalisation affine parameters and the fusion
gate are updated, episodically (reset after every batch), using three label-free losses: prior consistency, flip equivariance, entropy.

In [ ]:
tta, shift_det = T.build_tta(cfg, model, train_cache, device, shift_mode=cfg["tta_shift_mode"])
print("shift detector (clean source stats):", shift_det.state_dict())

# how well does the VQ shift score separate clean from degraded faces?
zs = {}
for deg in [None, ("occlusion", 3), ("blur", 3), ("lowlight", 3), ("weather", 3), ("compression", 3)]:
    ds = FaceDataset(test_cache, cfg, train=False, fixed_deg=deg, return_clean_targets=False)
    x = torch.stack([ds[i]["img"] for i in range(min(64, len(ds)))]).to(device)
    zs[str(deg or "clean")] = shift_det.z(x).cpu().numpy()
plt.figure(figsize=(10, 4))
for k, v in zs.items():
    plt.hist(v, bins=30, alpha=0.5, label=f"{k} (median z={np.median(v):.1f})")
plt.xlabel("shift z-score"); plt.legend(); plt.title(f"Shift detector ({cfg['tta_shift_mode']}): z-scored against clean training crops"); plt.show()

In [ ]:
# TTA vs no-TTA on the WFLW-C cells and on the real WFLW subsets.
#   gated  = PG-TTA as designed: episodic (reset after every batch), steps chosen by the shift detector
#   forced = same losses, always 2 steps, episodic                  -> "no shift gating" ablation
#   online = gated, but the adapted parameters carry over across the batches of a cell (reset per cell)
#            -> the standard corruption-stream protocol; adaptation accumulates instead of restarting
from armor.tta import PGTTA
FORCED_STEPS = 2
tta_forced = PGTTA(model, {**cfg, "tta_force_steps": FORCED_STEPS}, shift_det)
tta_online = PGTTA(model, {**cfg, "tta_online": True}, shift_det)
rows = []
print(f"{'cell':20s} {'NME':>7s} {'gated':>7s} {'forced':>7s} {'online':>7s} | {'FR':>6s} {'gated':>6s} {'forced':>6s} {'online':>6s} | {'z':>6s} {'steps':>5s}")
for deg in [None] + [(t, s) for t in DEG_TYPES[1:] for s in SEVERITIES]:
    r0 = T.evaluate(cfg, model, test_cache, device, fixed_deg=deg)
    r1 = T.evaluate(cfg, model, test_cache, device, fixed_deg=deg, tta=tta)
    r2 = T.evaluate(cfg, model, test_cache, device, fixed_deg=deg, tta=tta_forced)
    tta_online.reset()
    r3 = T.evaluate(cfg, model, test_cache, device, fixed_deg=deg, tta=tta_online)
    tta_online.reset()
    rows.append(dict(cell=str(deg or "clean"), NME=r0["full"]["NME"], NME_TTA=r1["full"]["NME"], NME_TTA_forced=r2["full"]["NME"], NME_TTA_online=r3["full"]["NME"],
                     FR=r0["full"]["FR"], FR_TTA=r1["full"]["FR"], FR_TTA_forced=r2["full"]["FR"], FR_TTA_online=r3["full"]["FR"],
                     z=r1["full"].get("tta_z", 0.0), steps=r1["full"].get("tta_steps", 0.0), vis_acc=r0["full"]["vis_acc"]))
    r = rows[-1]
    print(f"{r['cell']:20s} {r['NME']:7.3f} {r['NME_TTA']:7.3f} {r['NME_TTA_forced']:7.3f} {r['NME_TTA_online']:7.3f} | "
          f"{r['FR']:6.2f} {r['FR_TTA']:6.2f} {r['FR_TTA_forced']:6.2f} {r['FR_TTA_online']:6.2f} | {r['z']:6.1f} {r['steps']:5.2f}")
RESULTS["tta"] = rows
print("\nReading the table: 'z' = median shift score of the cell, z-scored against clean training crops (clean ~ 0); 'gated' spends steps only"
      " where z is high. 'forced' tells you whether the adaptation losses help at all; 'online' whether accumulating adaptation over a"
      " corrupted stream helps more than per-batch episodes. If 'forced'/'online' hurt, lower tta_lr (e.g. 3e-4).")
print("\nReal WFLW subsets with PG-TTA:")
res_tta_real = T.evaluate(cfg, model, test_cache, device, tta=tta, verbose=True)
RESULTS["wflw_test_tta"] = res_tta_real

## 9 · Ablations (optional — each call retrains stage B/C with one component removed)
Run the ones you need for the paper; in `FULL` mode each ablation costs about as much as the main run.

In [ ]:
def run_ablation(name, severities=SEVERITIES, **overrides):
    acfg = dict(cfg); acfg.update(overrides); acfg["out_dir"] = f"{cfg['out_dir']}/ablation_{name}"
    os.makedirs(acfg["out_dir"], exist_ok=True)
    p = T.train_stage_b(acfg, train_cache, stage_a, device, log=lambda *a: None)
    m = T.train_stage_c(acfg, train_cache, test_cache, p, device, log=lambda *a: None)
    tab = T.wflw_c_benchmark(acfg, m, test_cache, device, severities=severities, log=lambda *a: None)
    print(f"[{name}] clean NME {tab['clean'][0]:.3f} | mean corrupted NME {tab['mean_corrupted'][0]:.3f}")
    RESULTS.setdefault("ablations", {})[name] = tab
    return m

RUN_ABLATIONS = False
if RUN_ABLATIONS:
    run_ablation("no_reliability_supervision", w_rel=0.0, w_type=0.0)      # alpha learned implicitly (ORFormer-style)
    run_ablation("no_visibility_weighting", vis_floor=1.0, w_vis=0.0)      # plain STAR, no visibility head signal
    run_ablation("no_degradation_bank", p_degrade=0.0)                      # train on clean + geometric aug only

## 10 · Full pipeline on a real photo — SCRFD → crops → ARMOR landmarks → back-projection

In [ ]:
from armor.detector import download_scrfd, SCRFD, FacePipeline
onnx_path = download_scrfd(SCRFD_MODEL, root=f"{WORK}/models")
detector = SCRFD(onnx_path, input_size=640)
print("SCRFD:", onnx_path, f"{os.path.getsize(onnx_path)/1e6:.1f} MB | providers {detector.session.get_providers()}")

# pick an image: upload your own (Colab) or use a raw WFLW image
img_path = None
if IN_COLAB:
    from google.colab import files
    print("Upload a photo (or press cancel to use a WFLW image)")
    try:
        up = files.upload(); img_path = list(up.keys())[0] if up else None
    except Exception:
        img_path = None
if img_path is None and not USE_SYNTHETIC:
    img_path = os.path.join(f"{DATA}/WFLW_images", te_items[7]["name"])
if img_path is None:
    syn = make_synthetic_cache(1, 480, seed=5); img_path = f"{WORK}/synthetic_face.jpg"
    cv2.imwrite(img_path, cv2.cvtColor(np.asarray(syn["img"][0]), cv2.COLOR_RGB2BGR))

img_bgr = cv2.imread(img_path)
pipe = FacePipeline(detector, model, cfg, device)
pipe_tta = FacePipeline(detector, model, cfg, device, tta=tta)
t0 = time.time(); results = pipe.run(img_bgr); t1 = time.time()
if len(results) == 0:                                   # e.g. the synthetic cartoon face: retry with a permissive threshold
    results = pipe.run(img_bgr, score_thresh=0.15)
print(f"{len(results)} face(s) in {1000*(t1-t0):.1f} ms (first call includes warm-up)")
t0 = time.time(); results_tta = pipe_tta.run(img_bgr, score_thresh=0.15 if len(results) and results[0]["score"] < 0.5 else 0.5); print(f"with PG-TTA: {1000*(time.time()-t0):.1f} ms")
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(cv2.cvtColor(FacePipeline.draw(img_bgr, results), cv2.COLOR_BGR2RGB)); axes[0].set_title("ARMOR-FLD"); axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(FacePipeline.draw(img_bgr, results_tta), cv2.COLOR_BGR2RGB)); axes[1].set_title("ARMOR-FLD + PG-TTA"); axes[1].axis("off")
plt.show()
for r in results:
    print(f"box {np.round(r['box']).astype(int)} score {r['score']:.2f} | mean visibility {r['visibility'].mean():.2f} | mean patch reliability {r['reliability'].mean():.2f}")

## 11 · Size, memory and latency (Ch. 4 & 6 of the report) + ONNX export

In [ ]:
class ExportWrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        o = self.m(x); return o["coords"], torch.sigmoid(o["vis_logit"]), o["alpha"]

model.eval()
n_prior = sum(p.numel() for p in model.prior.parameters()) / 1e6
n_total = sum(p.numel() for p in model.parameters()) / 1e6
print(f"params: prior (encoder+R-ORFormer+codebook+decoder) {n_prior:.2f}M | landmark branch {n_total-n_prior:.2f}M | total {n_total:.2f}M")

export_path = f"{cfg['out_dir']}/armor_fld_{MODE.lower()}.onnx"
export_kw = dict(input_names=["crop"], output_names=["coords", "visibility", "unreliability"],
                 dynamic_axes={"crop": {0: "b"}, "coords": {0: "b"}, "visibility": {0: "b"}, "unreliability": {0: "b"}}, opset_version=17)
dummy = torch.randn(1, 3, cfg["crop_size"], cfg["crop_size"])
try:
    torch.onnx.export(ExportWrapper(model).cpu().eval(), dummy, export_path, dynamo=False, **export_kw)
except TypeError:   # older torch without the `dynamo` switch
    torch.onnx.export(ExportWrapper(model).cpu().eval(), dummy, export_path, **export_kw)
model.to(device)
print(f"ONNX landmark model: {os.path.getsize(export_path)/1e6:.1f} MB (fp32) | SCRFD: {os.path.getsize(onnx_path)/1e6:.1f} MB")

# latency: detector (640x640) and landmark net per face, batch 1 and batch 8
sess = ort.InferenceSession(export_path, providers=[p for p in ["CUDAExecutionProvider", "CPUExecutionProvider"] if p in ort.get_available_providers()])
x1 = np.random.randn(1, 3, cfg["crop_size"], cfg["crop_size"]).astype(np.float32); x8 = np.repeat(x1, 8, 0)
for _ in range(5): sess.run(None, {"crop": x1}); detector.detect(img_bgr)
def bench(fn, n=20):
    t0 = time.time(); [fn() for _ in range(n)]; return 1000 * (time.time() - t0) / n
print(f"SCRFD {SCRFD_MODEL} @640: {bench(lambda: detector.detect(img_bgr)):.1f} ms/image")
print(f"ARMOR landmark net ({sess.get_providers()[0]}): {bench(lambda: sess.run(None, {'crop': x1})):.1f} ms/face (b=1), "
      f"{bench(lambda: sess.run(None, {'crop': x8}))/8:.1f} ms/face (b=8)")
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        model(torch.randn(16, 3, cfg["crop_size"], cfg["crop_size"], device=device))
    print(f"peak GPU memory, fp16 inference, batch 16: {torch.cuda.max_memory_allocated()/1e6:.0f} MB")
RESULTS["size"] = dict(params_M=n_total, onnx_MB=os.path.getsize(export_path) / 1e6, scrfd_MB=os.path.getsize(onnx_path) / 1e6)

## 12 · Save everything

In [ ]:
with open(f"{cfg['out_dir']}/results_{MODE.lower()}.json", "w") as f:
    json.dump(RESULTS, f, indent=1, default=float)
print("saved", f"{cfg['out_dir']}/results_{MODE.lower()}.json")
print(json.dumps({k: RESULTS[k] for k in ["wflw_test", "wflw_c"] if k in RESULTS}, indent=1, default=float)[:3000])

## 13 · What to report (paper skeleton)

**Contributions.** (1) Degradation Bank with reliability supervision and the WFLW-C protocol; (2) R-ORFormer — reliability-supervised,
type-conditioned messenger recovery with reliability-biased attention; (3) reliability-gated prior fusion + visibility head + STAR-V;
(4) PG-TTA — prior-guided, source-free test-time adaptation gated by a VQ shift detector.

**Tables to fill from `results_full.json`.**
* Table 1 — WFLW test: NME / FR@0.1 / AUC@0.1 on *full* and the six subsets, next to the numbers reported in the ORFormer (WACV 2025) and STAR (CVPR 2023) papers.
* Table 2 — WFLW-C: NME per (degradation, severity) with and without PG-TTA; report mean-corrupted NME and the average number of TTA steps (cost).
* Table 3 — ablations: no reliability supervision, no visibility weighting, no degradation bank, no shift gating (set `tta_z_per_step=1e-9` → always `tta_max_steps`).
* Table 4 — efficiency: params, ONNX size, ms/face on T4 (fp16) and CPU, peak memory.

**Honest expectations.** `QUICK` numbers are only a plumbing check. `FULL` at 128² input with a 2-stack lite hourglass is a *small* model
(a few M parameters) trained for ~1 h, so expect it to trail the published ~4 % WFLW NME of 256²/4-stack models; to close that gap raise
`crop_size` to 256, `hg_stacks` to 4 and train longer on an A100 (`cfg = get_config("FULL", crop_size=256, heatmap_size=64, hg_stacks=4, c_epochs=120)`) —
the code is resolution-agnostic. None of the numbers in this notebook have been produced on WFLW yet: run `FULL` and report what you measure.
The claims that *are* defensible even at small scale are the robustness ones: the WFLW-C gap between the ablations and the full model, and the
TTA gain concentrated on high-shift cells while clean cells pay zero steps.